# `C08` — Iceberg regeneration

Scenario **C08** of the Track A certification ladder (card `OMIE-CERT-C08-1`, [`agile-plan.md`](../../docs/omiecid/agile-plan.md); canonical spec [`scenario-ladder.md`](../../docs/omiecid/scenario-ladder.md) §C8). Oracle kind: **exact-parametric, plus I3 and I6(b)**.

**Terms used below** (defined once in `scenario-ladder.md`, "What eager, jitted and assertion mean here"): *eager* runs every JAX operation immediately (`jax.disable_jit()`), so a human can inspect each step; *jitted* traces and compiles the whole run (`jax.jit`), the mode training uses; the two must agree. An *assertion* records a failure and the notebook carries on: every check is evaluated, and the verdict comes at the end. Every pass/fail assertion here is an **exact integer equality** (or membership in an explicitly listed set) on order ids, prices in ticks (0.01 EUR/MWh), quantities in ticks (0.1 MW), submission indices and the engine's stored book timestamps. Euro figures are shown, never asserted by this notebook (the battery's I6 holds the settlement report to its derived float tolerance); the one figure read back from a float that is asserted (the reported trade price of a single-price submission, converted to ticks) is held to a one-tick bracket and labelled `ENGINEERING INVARIANT`.

## 1. Purpose

> **K2 re-authored (r4, 2026-10-21; `OMIE-CERT-RERUN-3`).** One correction, from the amended specification only: `market-spec.md` v2.14 §4.2.1 Rule 2 *Incoming iceberg* items (2), (4) and (6), amended 2026-10-20 (`OMIE-SPEC-ICE-INCOMING-AMEND-1`); OQ-33 revised and OQ-34 resolved by the user's ruling of 2026-10-20; `scenario-ladder.md` §C8, amended 2026-10-20. **The incoming iceberg is a chain of instances of size $peak$.** Its trade rows split at resting entries **and** at its own instance boundaries: every row is at most $peak$, and every multiple of $peak$ below the swept quantity $M$ is a cumulative row boundary. A remainder $R > 0$ rests as instance $k_e = \lfloor M/peak \rfloor$ showing $v_1 = \min(peak - u, R)$, with $u = M - k_e\,peak$, and keeps $R - v_1$ hidden. This replaces the r2/r3 known result (one row per resting counter bid, never split at the incoming peak). The quotes of the superseded §4.2.1 sentences and of the OQ-33 2026-10-07 *Ruling* are retired and replaced by quotes of the amended texts. Findings F1 and F6 are rewritten; F14–F16 are added. **r4 addendum (same day, after the K3 generator variant):** a fourth construction, `C08.incoming_remainder` (`make_iceberg_cascade(seed, variant="incoming_remainder")`), leaves a remainder: its levels' total depth $M$ is strictly below the iceberg total and $M \bmod peak \ne 0$. On it (I) and (R) are asserted with $R > 0$ and $u > 0$, so the display $v_1$ is no longer asserted only vacuously (finding F14, revised). Everything else stands, including the r2 note below.

> **K2 re-authored (REPEAT, 2026-10-07; `OMIE-CERT-RERUN-1`, verdict log entry 21).** Two corrections, from the specification only. **(i) The incoming iceberg:** its trades are one per resting counter bid (OMIE Rules Annex §4.4; `market-spec.md` v2.14 §4.2.1 Rule 2, *Incoming iceberg*; OQ-33, RESOLVED), replacing the earlier known result that split its trades at its own slice boundaries. **(ii) The scope and the ceiling:** the effective iteration ceiling is **100 outer matching passes** (`_TRIP_BOUND`; `scenario-ladder.md` v2.4, *The capacity-validity precondition*, corrected by S7); the generator's hop envelope of 160 is not that ceiling. Carry-forward judging rules: a battery check reported `not_applicable` counts as vacuously satisfied, never as "not pass"; every recorded list is recorded in full; the eager runs are the first valid seeds of each construction.

**Purpose.** An iceberg shows a small peak and hides the rest; when the peak is consumed a new slice
appears and may immediately be consumed too. §4.2.1 Rule 4 makes a strong demand about this cascade:
it must reach a fixed point **inside the originating matching process**. Draining it across later
submissions would be a specification violation and would show up as a book left crossed between
arrivals — which is exactly what invariant I3 asserts. Rule 3 gets its own draw because it is the
genuinely awkward case: the aggressor only partially consumes the peak and rests, and the
*regenerated* slice must then cross that resting remainder and trade again.

Two precision points. The hop bound, not the nominal slice count, is what must stay under the
iteration ceiling — Rule-3 re-crossing adds hops, and a truncated draw cannot be distinguished from a
Rule-4 violation by looking at the trade list. Every iceberg also stays within the slice margin,
$\lceil \mathit{total}/\mathit{peak} \rceil \le 64$, because the same flag is raised at registration by any larger iceberg
([§ The capacity-validity precondition](#the-capacity-validity-precondition)). And the scenario must **not** assert anything about
internal `iceberg_meta` slot state: OQ-20 and OQ-23 rule that a stale slot is an engineering
invariant, explicitly **not** a Rule-4 breach, so asserting it would fail the scenario on a non-rule.

**What the caller is told.** The first attempt at this ladder found a real defect here: an
*incoming* iceberg that filled across several of its own slices wrote every slice's trade to the book
but reported only the first slice back to the caller — 30 MW traded, 10 MW and 125.00 EUR reported
instead of 375.00. It was fixed by `OMIE-ICE-INCOMING-FILL-REPORT-1` (merged 2026-09-28, `1f4755a`).
C8 therefore also certifies that the quantity and settlement a submission **reports** equal the trade
rows that submission **wrote** (invariant I6(b)), for incoming and resting icebergs alike. The claim
holds under four stated preconditions that I8 and the generator already guarantee: no truncation, no
book-full eviction, a sign-valid price increment, and a trade ring that is not saturated.

*(Verbatim from `scenario-ladder.md` §C8.)*

**Certifies** (card): §4.2.1 Rules 2/3/4 (Rule 2 including its v2.14 *Incoming iceberg* bullet, items (1)–(4) and (6), as amended 2026-10-20); §4.2 (*"a new slice … receives a new timestamp"*); §5.2, the iceberg-slice bullet ("Iceberg slices receive a new timestamp per slice, except …"; the passive-side exemption for price determination); §6.2 item 7. The Rule-3 trade price follows OQ-31, RESOLVED 2026-10-02 (Reading B): every trade prints at the regenerated slice's own limit, never at the aggressor's limit. Report-equals-tape is an **engineering invariant**, labelled as such. **Pass** (card): exact slice-size sequence and count; I3 at every return; hop bound asserted under the ceiling (the 100-pass bound); reported fill equals the tape rows written, exact in ticks. **Must not assert:** `iceberg_meta` slot cleanliness (OQ-20 / OQ-23).

**Scope, restated** (`scenario-ladder.md` v2.4 §C8, *Dataset*, with the v2.14 spec). Random `(total, peak)` with $\lceil total/peak \rceil \le 64$ (the C08 modes draw at most 39) and an **analytic worst-case hop bound under the 100-pass bound**. The generator's envelope alone admits a hop bound up to 160 (`CapacityEnvelope.max_hops`), so this notebook enforces the 100-pass bound itself: a draw whose analytic hop bound (the draw's, or the generic tape bound) exceeds 100 is outside the margin and is discarded before the run, and the aggregate assertion is hop bound ≤ 100 on every valid draw. Four draws: one aggressor consuming a resting iceberg whole (Rule 2, Rule 4); the separate Rule-3 draw; the separate incoming-remainder draw (below); and the separate incoming-iceberg draw: a chain of instances of size $peak$ whose trade rows split at resting entries **and** at its own instance boundaries (Rule 2, *Incoming iceberg*, item (2), amended 2026-10-20; OQ-33 revised). What C08 certifies about the incoming iceberg is therefore: it aggresses with its total; each row pairs one resting entry with one of its instances (every row at most $peak$, every multiple of $peak$ below $M$ a cumulative row boundary); each trade prints at the resting level's limit; it is filled exactly $total$; a remainder rests as instance $k_e$ showing $v_1$ (item (4); reached only by the incoming-remainder draw, finding F14); and its report equals the rows its submission wrote. The slice-size sequence (S) of a resting iceberg is asserted on the cascade and Rule-3 draws. On the incoming draw, (S) is the merge of its instance partition $(peak, \ldots, peak, r)$ with the resting-level partition.

**How the claim is pinned.** Four constructions of the generator, each on a fresh book of one contract, each in its own seed range (one seed names one draw):

- **`C08.iceberg_cascade`** (the cascade draw, `DRAWS` valid draws). Submission 0: a resting standard iceberg (PPD 0) of `total` with visible `peak`, $2 \le \lceil total/peak \rceil \le 39$. Submission 1: one NON aggressor of exactly `total`, two euros through the iceberg's limit. It must consume the whole iceberg, slice by slice, in its own submission.
- **`C08.rule3`** (the Rule-3 draw, `RULE3_DRAWS` valid draws). Submission 0: a resting standard iceberg with $n \ge 3$ slices. Submission 1: a NON aggressor with $peak < q_{agg} \le total - peak$ and a limit $L$ 1–200 ticks strictly through the iceberg's limit. After the visible peak is consumed the aggressor still has quantity; the regenerated slices (new-timestamp entries) must keep trading against it, whether the engine decomposes this as Rule 2 (the aggressor keeps matching) or Rule 3 (the remainder rests and the slice crosses it). Every trade prints at the slice's limit $p_{ice}$ (OQ-31), never at $L$.
- **`C08.incoming_iceberg`** (the incoming-iceberg draw, `INCOMING_DRAWS` valid draws). Submissions $0 \ldots L-1$: a ladder of $L \in [2, 8]$ resting NON levels on one side, at distinct prices. Submission $L$: an **incoming** iceberg on the other side, priced at the deepest level it needs, deep enough to fill all $\lceil total/peak \rceil$ of its own slices. It aggresses with its total as a chain of instances of size $peak$. Each trade row pairs one resting level (each NON level is one counter bid) with one of its instances, so a row ends where a level ends **or** where one of its instances ends and the next begins. Its report must equal the trade rows its submission wrote.
- **`C08.incoming_remainder`** (the incoming-remainder draw, `REMAINDER_DRAWS` valid draws; drawn with `make_iceberg_cascade(seed, variant="incoming_remainder")`, a variant outside `omie_tape_synth.CONSTRUCTIONS`). Submissions $0 \ldots L-1$: $L \in [2, 8]$ resting NON levels on one side, at distinct prices. Submission $L$: an incoming standard iceberg (PPD 0) on the other side whose limit crosses every level, with the levels' total depth below its total. It sweeps every level, $M$ = the total depth, as a chain of instances (rows split at levels and at instance boundaries), and its remainder $R = total - M$ rests as instance $k_e = \lfloor M/peak \rfloor$ at $p_{ice}$, showing $v_1 = \min(peak - u, R)$ with $u = M - k_e\,peak$, and $R - v_1$ hidden. Everything is derived from the rule; the generator's `expected` is not read.

Per valid draw, in one combined assertion whose expected and observed values are recorded **in full** (section 7): **(A)** acceptance; **(Q)** the exact trade sequence, row by row: buyer order id, seller order id, price, quantity, submission (and, on the incoming draw, the price-setting resting order); **(S)** on the cascade and Rule-3 draws, the slice-size sequence and count it implies (on the incoming draw, the (counter bid, instance) partition: the merge of the level partition with $(peak, \ldots, peak, r)$, summing to $total$); **(I)** on the incoming draw, the explicit instance-split check: no row of its submission above $peak$, and no multiple of $peak$ below $M$ missing from the cumulative row boundaries; **(R)** on the incoming draw, the rested remainder `[price, visible, hidden]` $= [p_{ice} + k_e\,ppd,\ v_1,\ R - v_1]$, or none when $R = 0$; **(F)** every order's filled and remaining quantity, and no eviction or cancellation; **(B)** the final book (exact, or one of an explicitly listed set where finding F4 applies); **(N)** on the Rule-3 draw, the resting slice's engine timestamp strictly later than the first slice's; **(E)** the engine's report per submission, `[accepted, traded, quantity in ticks]`, equal both to the expectation and to the sum of the trade rows that submission wrote (I6(b), an engineering invariant). Then the battery (RB, I1–I9) on every run, with explicit assertions that I3 and I6 hold on every valid draw (`pass`, or `not_applicable` counted as vacuously satisfied; only `fail` fails), and the capacity checks (hop bound ≤ 100 passes, largest slice count, no flag raised inside the margin).

**K2 findings** (recorded here; none blocks the expectation; F1, F2, F4, F6 and F7 revised and F11–F13 added at the 2026-10-07 re-authoring):

- **F1 — spec version.** The card names `market-spec.md` v2.12 as the default quote source; the file is **v2.14**. v2.13 changed §4.2.1 Rule 3 (the *Trade price of the hop* bullet, Classification) and §6.2 item 7 (OQ-31). v2.14 (2026-10-07) was additive (new dated bullets), adding to §4.2.1 Rule 2 the *Re-ranking* and *Incoming iceberg* bullets, to Rule 3 the *ICE aggressor* bullet, and to Rule 4 the *Trade timestamps* and *Slice activation timing* bullets. Its amendment of 2026-10-20 (`OMIE-SPEC-ICE-INCOMING-AMEND-1`), kept under the v2.14 token, is **not** additive: it rewrote *Incoming iceberg* items (2), (4), (6) and its *Classification*, rewrote the first sub-bullet of Rule 3's *ICE aggressor* bullet, and deleted the superseded sentences. The r3 quotes of the old item (2) no longer exist in the file; they are retired, and the amended texts are quoted instead. Every quote is checked against the current file at run time.
- **F2 — "all trades carry the originating matching process's timestamp" is certified as attribution to the originating submission.** v2.14 §4.2.1 Rule 4, *Trade timestamps*, says that parenthetical "overstates the source": every cascade trade is "concluded within, and attributed to, the originating matching process", and the trades "need **not** share one identical timestamp value" (Matching Algorithm §5.3). So equal trade-row timestamps are **not** asserted (the number of distinct ones is reported). What is asserted is the rule's observable content: every trade of the draw is written **during the aggressor's own submission** (the submission index of each trade row, in (Q)), none is missing at its return (Q), and the book is never crossed at a return (I3). In the cascade and Rule-3 tapes the aggressor is the last submission, so a cascade drained "across submit cycles" would leave trades missing and the final book crossed, both of which fail.
- **F3 — passive/aggressor labels on a resting iceberg's fills are not asserted.** Which leg the engine *labels* passive at a Rule-2 or Rule-3 fill is the open engine-facing item (b) of OQ-31 (and the review's §16.2 note on D3-07); the rules fix the price-setter (the iceberg), not the label. So the cascade and Rule-3 trades are pinned by **buyer and seller order ids** (fixed by the sides), price, quantity and submission; the number of hop-shaped rows (passive = the aggressor's own id) is reported, not asserted. On the incoming draw the label is rule-determined and pinned: the resting NON level is the order already in the book (§6.2 item 3).
- **F4 — a just-exhausted slice may be absent between aggressors.** When the aggressor ends exactly at a slice boundary (Rule-3 draws with $q_{agg} \bmod peak = 0$), the Matching Algorithm §3.3 shows the next slice at once, while §4.2.1 Rule 4's *Scope clarification* and OQ-20 permit the next slice to be activated later ("a permitted between-aggressor state"). The final book is therefore asserted to be **one of** two listed books: the iceberg's next slice visible at $p_{ice}$ with $\min(peak, total - q_{agg})$, or no visible row for it. In both, the iceberg's remaining quantity (visible + hidden) is asserted exactly: $total - q_{agg}$. Nothing about `iceberg_meta` slots is asserted (card, *Must not assert*). *v2.14 note:* Rule 4's *Slice activation timing* correction says activation is **eager** in the market (only the first alternative of the Scope clarification has source support), but leaves "whether a lazily activated slice makes any observable difference" — its example is "a resting book snapshot that lacks the slice" — as an engineering question for the architect, and does not reopen the OQ-20/OQ-23 conclusions. The re-authoring was scoped to the two corrections above, so the two-book set is kept and recorded as finding F13.
- **F5 — the Rule-3 draw certifies the outcome, not a separate hop event.** v2.13 Rule 3: "In the real algorithm this hop is not a separate event." The reference walk (W) runs the real algorithm (the aggressor keeps matching while it is executable, §5.1.3 of the Matching Algorithm); the Rule-3 decomposition produces the same trades (same sizes, same counterparties, same submission) and, under OQ-31, the same price. The expectation is identical under both, so it does not depend on which one the engine uses.
- **F6 — the incoming iceberg: a chain of instances** (re-authored at r4; v2.14 §4.2.1 Rule 2, *Incoming iceberg*, items (1)–(4) and (6), amended 2026-10-20; OQ-33 revised and OQ-34 resolved 2026-10-20). An incoming iceberg with a competitive price aggresses with its **total** quantity (item (1)) as a chain of instances of size $peak$, the last one smaller (item (2)). Each trade is between one resting counter-bid entry (each NON level is one) and one incoming instance. So a row ends where a level ends **or** where an instance ends and the next begins, and no row spans two resting entries. Each trade prints at the resting entry's limit (item (3); §6.2 item 3). Instance $k$ trades only where its own limit $b + k\,ppd$ crosses (item (6)); at PPD 0 that limit is $p_{ice}$ for every instance. So the trade sequence is the merge of the level partition with $(peak, \ldots, peak, r)$, in price priority, until $total$ is filled. Every row is at most $peak$, and every multiple of $peak$ below $M$ is a cumulative row boundary. This replaces the r2/r3 reading (one row per level, never split at the incoming peak), which the 2026-10-20 ruling supersedes. The card's "reported quantity = Σ slice fills = $total$" now reads literally: the rows each instance took, summed over the instances, = Σ trade rows of the submission = $total$ = the reported quantity. **No single-timestamp claim** is made about these trades: each slice is a new-timestamp entry (§4.2, §5.2), and the number of distinct trade-row timestamps is reported.
- **F7 — a flag raised inside the margin is a failure, not only a discard.** The harness discards a draw that I8 marks `invalid` (it counts in the discard rate). The card's Known result also lists `truncated = False`, and the capacity-validity precondition says the generator guarantees the margin and the flags only confirm it. So a draw that passed the pre-run margin check (hop bound ≤ 100 passes and within the generator envelope, ≤ 80 trades, ≤ 80 resting per side, ≤ 800 icebergs, every $\lceil qty/peak \rceil \le 64$) and still raised a capacity flag is discarded **and** recorded as a failed assertion: on this scenario a truncation inside the analytic bound is indistinguishable from a non-terminating Rule-4 cascade.
- **F8 — what the engine reports.** The spec defines no per-submission reported price or settlement. Report-equals-tape (I6(b)) is asserted here on the quantity, exactly in ticks: the reported quantity equals the trade rows the submission wrote **and** the expected traded quantity. The settlement half is the battery's I6 (float tolerance, V10); its euro figures are recorded per draw, not asserted here. The reported price is asserted only where a submission traded at one price (cascade and Rule-3 aggressors): within one tick of $p_{ice}$, an `ENGINEERING INVARIANT`. On the incoming draw it is recorded, not asserted.
- **F9 — no competition at one price.** Each draw has one iceberg and, on the incoming draw, resting levels at pairwise distinct prices, so the two-iceberg hazard of D3-08 cannot arise. Every trade sequence is still pinned exactly, row by row.
- **F10 — the new-timestamp claim, where it is observable.** On the Rule-3 draw at least one slice is exhausted ($q_{agg} > peak$), so any slice of the iceberg resting at the end is a later slice; by §4.2 and §5.2 ("unique, sequential") its engine timestamp `(sec, nsec)` must be strictly later than that of the first slice, read after submission 0. In the cascade nothing of the iceberg remains, and on the incoming draw nothing of it rests, so the claim is not observable there and is not asserted.
- **F11 — the ceiling, restated (S7).** The iteration ceiling is the fixed **100 outer matching passes** (`_TRIP_BOUND`); `max_match_iterations` = 200 never binds (`scenario-ladder.md` v2.4, *The pass ceiling*). The pass count cannot be read after a run, so the hop bound is the **analytic** guarantee and `truncated == False` the confirmation. "Under the 100-pass bound" is read as **hop bound ≤ 100**: the hop bound is a worst-case count of passes needed, and 100 passes that finish the quantity leave nothing for `truncated` to flag. A draw with a hop bound in $(100, 160]$ passes the generator envelope but is outside the margin here: discarded before the run, never run, counted in the discard rate.
- **F12 — judging rules carried forward.** A battery check reported `not_applicable` is vacuously satisfied (the I3/I6 aggregate fails only on `fail`); every expected and observed list (trade rows, failing seeds, departures, eager seeds) is recorded in full, never sliced; the eager runs are the first `EAGER_DRAWS` **valid** seeds of each construction (plus the first valid draw of each class).
- **F13 — a just-exhausted slice under v2.14 (open for the orchestrator).** Under the *Slice activation timing* correction, the market shows the next slice at once, which would make F4's "no visible row" alternative a departure from the market. The same bullet routes the observable consequence of lazy activation to the architect and does not reopen OQ-20/OQ-23. This K2 keeps the two-book set of F4 (it was not among the corrections ordered) and records that tightening it to the visible-slice book alone is a spec-derivable option that needs a decision.
- **F14 — the remainder clause (item (4)): reached by the incoming-remainder draw only** (revised at the r4 addendum). On every `C08.incoming_iceberg` draw $M = total$ and $R = 0$, by construction: the levels hold at least $total + 1$ tick, $p_{ice}$ is the deepest level the sweep needs, and at PPD 0 every instance's limit is $p_{ice}$. There (R) asserts "no rested remainder". The `C08.incoming_remainder` draw has $M < total$ and $M \bmod peak \ne 0$, so $R > 0$ and $u > 0$, the decisive case of OQ-34's *Display C3*. On it (R) asserts the rested instance `[price, visible, hidden]` $= [p_{ice} + k_e \cdot 0,\ v_1,\ R - v_1]$, derived from the rule in (W) and in the closed form, never from the generator's `expected`. An aggregate assertion requires at least one valid draw with $R > 0$ and $u > 0$. The numbers of draws with $R > 0$, $u > 0$ and $v_1 < R$ are reported. Results-JSON note: `Certification.emit` takes generator digests only of keys in `omie_tape_synth.CONSTRUCTIONS`, which does not hold this variant, so its digest over seeds 0–7 is computed here the same way and recorded in that aggregate assertion's detail.
- **F15 — item (6) at PPD 0.** Every C08 iceberg has PPD 0, so each instance's limit is $b$, and "the sweep stops at the first instance that cannot trade" coincides with the ordinary stop at the first level that does not cross. The per-instance ICEwPI limits are not certified here. OQ-35 (a drifted price beyond ±9,999 EUR/MWh) does not arise at PPD 0.
- **F16 — what discriminates the amended rule, and the data-derived display.** An aggregate assertion requires at least one valid incoming draw in which one level's fill is split at an incoming instance boundary (more rows than levels touched). On such a draw the superseded r3 known result (one row per counter bid) is false, so the check is not vacuous; the number of such draws is reported. The $v_1$ display for a part-used instance ($u > 0$) is, by the spec's own label, "derived from OMIE data, not stated by the sources" (item (4); OQ-34 *Display C3*). It is quoted as such and, per F14, not exercised.


## 2. The rule, quoted

The governing passages, quoted **verbatim** (byte-copied) and checked against the document each names at run time (`notebooks.certification.cert_support.check_quotes`). A quote that has drifted is flagged **DRIFT** below and recorded as `quote_drift`: the notebook still runs to the end and ends `NOT CERTIFIED`. The regulatory sources are shown as citations, not machine-checked.

How the quotes ground the expectation of section 5:

- **What an iceberg is (§4.2, §4.5).** A total, a visible peak, and a new slice, with a new timestamp, each time a slice is exhausted; what is not consumed stays in the book.
- **One aggressor, many slices (§4.2.1 Rule 2; §6.2 item 7).** The aggressor fills slice after slice within its own submission; the iceberg sets the price of every slice.
- **The Rule-3 configuration and its price (§4.2.1 Rule 3, v2.13; OQ-31 Resolution and Consequences).** The aggressor's remainder can trade with a regenerated slice; the price is the slice's own limit ($p_{ice}$ for PPD 0), never the aggressor's limit $L$, which is drawn strictly through $p_{ice}$ so that the two readings differ.
- **The fixed point (§4.2.1 Rule 4, in full).** The whole cascade resolves inside the originating matching process; "a crossed order book cannot persist" (I3); no claim on internal slot state (Scope clarification; OQ-20 Ruling; OQ-23 Q2).
- **Timestamps (§5.2).** A new slice gets a new timestamp from the single, sequential queue; for price determination the iceberg is the passive side throughout.
- **The incoming iceberg (§4.2.1 Rule 2, *Incoming iceberg*, v2.14 as amended 2026-10-20; Rule 3, *ICE aggressor*; OQ-33 revised and OQ-34 resolved 2026-10-20; §6.1, §6.2 items 2, 3, 6).** It aggresses with its total as a chain of instances of size $peak$. It records one trade per (counter bid, incoming instance) pair at the counter bid's limit, level after level, so its rows split at resting entries and at its own instance boundaries. A remainder would rest as instance $k_e$ showing $v_1 = \min(peak - u, R)$ (findings F6, F14, F16).
- **Ranking of a regenerated slice (§4.2.1 Rule 2, *Re-ranking*, v2.14).** Each new slice is ranked as a new entry; the aggressor always trades the best-ranked counter bid next (the reference walk re-ranks after every activation).
- **Trade timestamps and slice activation (§4.2.1 Rule 4, v2.14 bullets).** Attribution to the originating process, not one shared timestamp value (finding F2); activation is eager in the market (findings F4, F13).
- **The ceiling** is not a quotable source passage here (`scenario-ladder.md` is not a checked quote source): it is restated in section 1 (finding F11) from `scenario-ladder.md` v2.4, *The capacity-validity precondition*.
- **The domain review (§17.3, D3-08).** The D3 gate of this K2, and why every counterparty sequence is pinned exactly.

In [ ]:
import time

T0 = time.time()   # the measured wall clock (results: wall_clock_s) starts here

from IPython.display import Markdown, display

from notebooks.certification.cert_support import (
    Certification, Citation, SpecQuote, check_quotes, render_quotes,
)

QUOTES = (
    # --- §4.2 / §4.5: what an iceberg is, and that each new slice gets a new timestamp ---
    SpecQuote(
        'market-spec.md',
        '§4.2',
        'A regular limit order with hidden total quantity and visible peak quantity.',
        note='The iceberg: a total quantity, of which only a peak is shown.',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2',
        'Only the peak size is shown in the order book.',
        note='What the book exposes: the visible slice only (the final-book assertion reads visible rows).',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2',
        'When a slice is exhausted, a new slice of peak size is activated and receives a new timestamp.',
        note="Slice sizes peak, ..., peak, r; each new slice is a new-timestamp entry (asserted on the Rule-3 draw's resting slice, finding F10).",
    ),
    SpecQuote(
        'market-spec.md',
        '§4.5',
        '| ICE | Yes (peak slices) | Yes (hidden qty rests) | Iceberg only |',
        note='An iceberg that is not consumed whole keeps its remainder (visible + hidden) in the book.',
    ),
    # --- §4.2.1 Rule 2: one aggressor, many slices, one call ---
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        '**Rule 2 — Multi-slice matching per aggressive order (NON, IOC, FOK; M6.4 update).**\nA single aggressive order (NON, IOC, or FOK) can fill multiple iceberg slices within one match cycle. After the visible peak is consumed, the iceberg regenerates a new slice (at its natural drifted price for ICEwPI)',
        note='The cascade draw: one aggressor of size total consumes all ceil(total/peak) slices in its own submission. (part 1 of 2; the passage is quoted in full, in consecutive parts)',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        "and matching continues against the same aggressive order until either the aggressive order is fully filled, the iceberg is fully exhausted, or the regenerated slice's price no longer crosses the aggressive order's limit. All slice interactions occur within a single `submit_order` call.",
        note='(continued, part 2 of 2)',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        "*Consequence:* To consume a 10 MW standard iceberg (PPD=0) with peak = 3 MW, a single NON, IOC, or FOK order for 10 MW fills all four slices (3 + 3 + 3 + 1 MW) within one match cycle. For ICEwPI icebergs, matching stops at the slice whose drifted price no longer crosses the aggressive order's limit.",
        note='The worked example of the slice sequence peak, ..., peak, r (here 3, 3, 3, 1).',
    ),
    # --- §4.2.1 Rule 2, v2.14: re-ranking and the incoming iceberg (OQ-33) ---
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        'Subject to §6.1, each regenerated slice is ranked as a new entry at its own (drifted) price. The aggressor always trades the best-ranked counter bid in the book next.',
        note='Re-ranking (v2.14): the reference walk re-selects the best-ranked counter bid after every slice activation.',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        'Rule 2 above names NON, IOC and FOK aggressors. An incoming iceberg (ICE) with a competitive price is also an aggressor, and these rules apply to it:',
        note='The incoming-iceberg draw (v2.14 Incoming iceberg bullet; finding F6).',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        '**(1) Total quantity.** It aggresses with its **total** quantity, not only its visible peak. It keeps matching while it remains executable and has quantity left.',
        note='The incoming iceberg sweeps its total, not one peak at a time.',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        '*(amended 2026-10-20, `OMIE-SPEC-ICE-INCOMING-AMEND-1`, user ruling 2026-10-20: items (2), (4) and (6) and the classification state the instance chain; history in the v2.14 row, OQ-33 and OQ-34).*',
        note='The 2026-10-20 amendment of the Incoming iceberg bullet: the text this r4 re-authoring follows (findings F1, F6).',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        "Below, $b$ is the incoming iceberg's price, $P$ its peak (reduced part), $d$ its price increment (PPD; $d = 0$ for a standard iceberg), $M$ the quantity it sweeps on entry and $R$ its remainder.",
        note='The notation: in this notebook b = p_ice, P = peak, d = ppd = 0.',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        '**(2) One transaction per (counter bid, instance) pair.** The incoming iceberg is a chain of instances, each of size $P$; the last instance may be smaller. Each trade is between one resting counter-bid entry and one incoming instance.',
        note='The incoming draw: the trade rows are the merge of the level partition with the instance partition (peak, ..., peak, r). (part 1 of 2; the passage is quoted in full, in consecutive parts)',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        'Each resting NON order is one counter bid, and each instance of a resting iceberg is its own counter bid, with its own creation time. A trade row ends where a resting entry ends **or** where one incoming instance ends and the next begins. Two resting instances are never merged into one row. (OQ-33, revised 2026-10-20.)',
        note='(continued, part 2 of 2) The (I) check: no row above peak, every multiple of peak below M a cumulative row boundary.',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        "**(3) Price.** Every trade prices at the resting entry's limit (§6.2 item 3).",
        note="Each incoming-draw trade prints at the resting level's limit.",
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        '**(4) Remainder.** If a remainder $R > 0$ is left once the order is no longer executable, it enters the book **as an iceberg** with total $R$. With $k_e = \\lfloor M/P \\rfloor$ and $u = M - k_e P$ (the part of instance $k_e$ already traded), the remainder keeps its place in the ladder: it continues as instance $k_e$, at limit $b + k_e\\,d$.',
        note='The (R) expectation: a remainder continues as instance k_e at p_ice + k_e * ppd (finding F14: R = 0 on every C08 draw). (part 1 of 3; the passage is quoted in full, in consecutive parts)',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        'It shows $v_1 = \\min(P - u, R)$, the unused part of instance $k_e$, and keeps $R - v_1$ hidden. This is "at most the reduced part … even if the matched quantity was not a multiple of the reduced part" (Annex §4.4).',
        note='(continued, part 2 of 3) The display v_1 and the hidden R - v_1 of (R).',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        'As for a NON residual, partial execution assigns no new timestamp (§5.2). The display after a part-used instance ($u > 0$) is derived from OMIE data, not stated by the sources (OQ-34 resolution).',
        note='(continued, part 3 of 3) The u > 0 display is data-derived, labelled as such (finding F16).',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        '**(6) Instance limits.** Instance $k$ ($k = 0, 1, 2, \\dots$) trades only at prices its own limit $b + k\\,d$ crosses; for $d = 0$ every instance has limit $b$. The sweep stops at the first instance that cannot trade, even if $b$ would still cross.',
        note='The walk checks each instance against its own limit; at PPD 0 every instance has limit p_ice (finding F15).',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        '*Classification:* **Consistent with market rules** for (1)–(6) (user ruling 2026-10-20; OMIE data; Annex §4.4–§4.5).',
        note="The amended bullet's classification. (part 1 of 2; the passage is quoted in full, in consecutive parts)",
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        '*Simulator limit:* on a trade ring without room for the split rows of item (2), the rows are left unsplit (volumes and prices unchanged; a certificate gap, not a market rule).',
        note='(continued, part 2 of 2) Why the unsaturated-ring precondition (I8) matters for the split: inside the margin the rows must be split.',
    ),
    # --- §4.2.1 Rule 3 and its trade price (v2.13, OQ-31 Reading B) ---
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        "**Rule 3 — Post-match slice regeneration crossing.**\nWhen a NON aggressive order partially fills an iceberg's visible peak and the aggressive order's unfilled remainder rests in the book, the regenerated iceberg slice (submitted during post-match regeneration within the same submit cycle) may cross that resting remainder.",
        note='The Rule-3 draw: the aggressor is larger than the peak and smaller than total - peak, so the regenerated slices must keep trading against it. (part 1 of 2; the passage is quoted in full, in consecutive parts)',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        'The resting remainder acts as passive liquidity for the new slice, producing an additional trade. This is correct simulator behavior consistent with how any new order (including a regenerated slice) triggers regular matching against the current book state.',
        note='(continued, part 2 of 2)',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        '*Trade price of the hop (clarification, v2.13, OQ-31 RESOLVED 2026-10-02):* "passive liquidity" above describes **matchability only**. The resting remainder can be matched by the regenerated slice. It does **not** set the trade price. The hop trades at the **iceberg slice\'s own limit price**:',
        note="The ruling the Rule-3 draw certifies: every trade prints at the slice's own limit, p_ice_int (PPD = 0), never at the aggressor's limit L. (part 1 of 2; the passage is quoted in full, in consecutive parts)",
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        "the base price for PPD = 0, or the slice's drifted price $\\mathit{base} + k\\,\\mathit{ppd}$ for ICEwPI. It never trades at the aggressor's limit $L$. For price determination the iceberg is the passive (resting) side of the hop, as in §5.2 and §6.2 item 7. If the regenerated slice's price does not cross $L$, there is no hop trade (§6.2 item 2; Rule 2).",
        note='(continued, part 2 of 2)',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        "In the real algorithm this hop is not a separate event. It is one more slice inside the aggressor's single matching process. The remainder-rests-then-slice-crosses sequence is the simulator's decomposition of that process.",
        note='Why the expectation does not depend on the decomposition (finding F5): the outcome is one matching process of the aggressor.',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        "*Classification:* **Consistent with market rules.** The regenerated slice is a new-timestamp entry and legitimately matches against any resting order. At this hop the trade price is the iceberg slice's limit (OQ-31).",
        note="Rule 3's classification, aligned with OQ-31.",
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        'The incoming iceberg aggresses with its total quantity as a chain of instances: instance $k$ trades only where its own limit $b + k\\,d$ crosses (Rule 2, *Incoming iceberg*, items (2) and (6)). Only its non-executable remainder rests, and only after the sweep ends, at the rest price $b + k_e\\,d$ with $k_e = \\lfloor M/P \\rfloor$ (item (4))',
        note='Rule 3, ICE aggressor (first sub-bullet, rewritten 2026-10-20): the incoming iceberg rests only after its sweep, at b + k_e d.',
    ),
    # --- §4.2.1 Rule 4, in full ---
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        '**Rule 4 — Regen-immediate-rematch cascade resolves within the originating matching process (M; P2-PERF-8 ruling, 2026-05-26).**\nWhen a regenerated iceberg slice (Rule 2/Rule 3)',
        note="The fixed point inside the originating matching process: every trade is written during the aggressor's own submission, and the book is never left crossed (I3). (part 1 of 3; the passage is quoted in full, in consecutive parts)",
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        "is **itself** crossed by still-resting opposite-side liquidity present in the book — and when the resulting trade exhausts that new peak and triggers a *further* regeneration that again crosses, and so on — **the entire cascade MUST resolve to a fixed point within the originating matching process** (all resultant trades carry that matching process's timestamp;",
        note='(continued, part 2 of 3)',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        'the order book is never left in a crossed state between aggressor arrivals). It is **spec-violating** to leave the regenerated slice resting visibly in a crossed book until the next incoming aggressive order triggers matching ("draining the cascade across submit cycles").',
        note='(continued, part 3 of 3)',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        '*Operative regulatory text (Continuous Trading Matching Algorithm):*',
        note="Rule 4's operative text follows (four sub-bullets).",
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        '§4.1, "Order book": the order book "is updated whenever orders are added to it, deleted from it, modified (volume, price, execution restriction, **activation of a new iceberg slice**)... **After each order book update...',
        note='Slice activation is a book update; any crossing it creates is concluded at once. (part 1 of 2; the passage is quoted in full, in consecutive parts)',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        'the algorithm validates if new matching opportunities have arisen and, if so, the resulting trades are concluded.**" — Slice activation is itself a book-update event, and any crossing it creates is concluded at once, not deferred.',
        note='(continued, part 2 of 2)',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        '§5.1: "Regular matching is triggered by the entry of an order with a new time stamp. An order with a new time stamp may be a newly entered order, a modified order, a (re)activated order that was inactive before, **or a new slice of an iceberg order**." — Each regenerated slice is an independent regular-matching trigger.',
        note='Each new slice is a regular-matching trigger.',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        '§5.1.3: an order with a new timestamp "continues to be matched against orders entered at this price level. This process continues **as long as the incoming order remains executable and has a positive order quantity.**" — Matching runs to a fixed point within the process; a partially-crossing remainder does not pause.',
        note='Matching runs to a fixed point within the process.',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        '§5.1.2: the iceberg\'s price governs the trade "**even if the timestamp of the iceberg order is renewed during the matching process**" — explicitly contemplates renewed slice timestamps *within a single matching process*, confirming intra-process resolution.',
        note="The iceberg's price governs across renewed slice timestamps.",
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        '*Why across-cycle draining is observably distinguishable (i.e. why this is M, not merely E):* In **regular** (continuous) trading a crossed order book cannot persist. Only **batch matching** (§5.2), triggered solely by a cross-border capacity update, allows a previously-crossed book to wait for a separate matching process; there is no analogous deferral in regular matching.',
        note='"a crossed order book cannot persist": the I3 claim, and the three observables a drain would show (crossed snapshot, wrong triggering submission, reordered trades). (part 1 of 2; the passage is quoted in full, in consecutive parts)',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        'Leaving a regenerated slice resting in a crossed book until the next aggressor would (a) expose a crossed LOB snapshot to any observer reading the book between aggressor arrivals, (b) emit the cascade trades under a later (wrong) triggering timestamp, and (c) reorder the trade sequence relative to any orders that arrive in between. All three are observable.',
        note='(continued, part 2 of 2)',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        '*Scope note vs. Rule 3:* Rule 3 covers the single-hop case where the new slice crosses the *resting remainder of the same aggressor*. Rule 4 generalizes to the multi-hop cascade where the new slice crosses *any* resting opposite-side liquidity and may itself trigger further regenerations — the fixed-point requirement applies to the whole chain.',
        note='Rule 3 is the single-hop case; Rule 4 the whole chain.',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        '*Scope clarification — what Rule 4 governs vs. what it does NOT (P2-PERF-12 ruling, 2026-05-28).* Rule 4 is a **within-call (intra-process) fixed-point requirement** on the resting **order book** (`OrderBook` / `lob`). It is **not** a post-call cleanliness invariant on simulator-internal side-channel state (e.g. `iceberg_meta`).',
        note='Why no iceberg_meta slot state is asserted, and why a just-exhausted slice may be absent from the book between aggressors (finding F4). (part 1 of 6; the passage is quoted in full, in consecutive parts)',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        'The post-condition the spec requires after a `submit_order` (or any other matching-triggering event) returns is exactly this: the resting order book must not be crossed, and no regenerated visible peak that crossed during the call may have been left visibly resting in a crossed book ("draining across submit cycles" — explicitly forbidden, see above).',
        note='(continued, part 2 of 6)',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        'The spec does NOT require that every slot in a simulator-side bookkeeping array describing icebergs hold an `order_id` currently present in the resting book;',
        note='(continued, part 3 of 6)',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        "an iceberg whose visible peak was fully consumed by a prior aggressor's matching process and whose next slice has been registered for later regeneration is a permitted between-aggressor state under the matching algorithm (a new slice is activated when the previous one's quantity reaches zero or when subsequent matching draws against the remaining hidden quantity — see",
        note='(continued, part 4 of 6)',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        'Continuous Trading Matching Algorithm §3.3, lines 265–275). The "no `peak_consumed=True` resting slot" assertion is a **simulator implementation invariant** of the current `iceberg_manager` design, **not** an OMIE market rule. The spec requirement is: every regenerated slice that crossed during the matching process must have been fully consumed (zero remaining visible)',
        note='(continued, part 5 of 6)',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        'or fully rested (positive remaining visible at a non-crossing price) by the time the matching process returns. A side-channel slot whose `order_id` is absent from the LOB is consistent with the spec when the slot represents an iceberg whose previous peak was fully consumed and whose next peak has not yet been activated.',
        note='(continued, part 6 of 6)',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        '*Classification:* **Consistent with market rules — within-process (within-call) resolution is mandated on the resting order book.** The scope clarification above is reaffirmed (not amended) market semantics:',
        note="Rule 4's classification, with the OQ-20 / OQ-23 disposition. (part 1 of 5; the passage is quoted in full, in consecutive parts)",
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        'post-call invariants on simulator-internal bookkeeping are an engineering concern, governed by Rule 4 only insofar as they cause an observable difference in the resting order book or the trade sequence. See `docs/omie/sprint-status.md` (P2-PERF-8 domain ruling 2026-05-26 and P2-PERF-12 scope ruling 2026-05-28) for the implementation history;',
        note='(continued, part 2 of 5)',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        'see also OQ-20 for the engineering-invariant tracking entry. **OQ-23 (2026-07-09)** dispositions a second, distinct route to the same OQ-20-class state — a `dirty`-invisible cross-slot peak consumption inside `regenerate_slices` that leaves a lower-index iceberg slot with a consumed visible peak at pass end;',
        note='(continued, part 3 of 5)',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        'it is **CONSTRUCTIBLE on valid inputs but is NOT a Rule 4 violation** (the resting book remains uncrossed and all trades are emitted intra-call — the surviving state is a permitted between-aggressor stale side-channel slot per this Scope clarification), and it warrants a scoped **engineering** follow-up (not a market-spec change)',
        note='(continued, part 4 of 5)',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        'because the P2-PERF-8 `no_fill_path` regen-skip optimization depends on the broken post-condition. See OQ-23 for the full ruling and a constructibility probe.',
        note='(continued, part 5 of 5)',
    ),
    # --- §4.2.1 Rule 4, v2.14 bullets: trade timestamps and slice activation timing ---
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        "Every trade of the cascade is **concluded within, and attributed to, the originating matching process**. No cascade trade may be stamped with the arrival of a later aggressor; that is the across-cycle defect forbidden above.",
        note='What finding F2 asserts: attribution of every trade to the aggressor\'s own submission.',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        "The cascade's trades need **not** share one identical timestamp value.",
        note='Why equal trade-row timestamps are reported, never asserted (finding F2).',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        'Only the first alternative has source support. Activation is **eager**.',
        note='Slice activation timing (v2.14): findings F4 and F13.',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        'Whether a lazily activated slice makes any observable difference is an engineering question for the architect.',
        note='Why the F4 two-book set is kept at this K2 and surfaced as finding F13.',
    ),
    # --- §5.2 timestamps and §6.1 / §6.2 regular matching ---
    SpecQuote(
        'market-spec.md',
        '§5.2',
        'A **new timestamp** is assigned on: modification (price, quantity, execution restriction), reactivation from inactive, or activation of a new iceberg slice.',
        note='A new iceberg slice gets a new timestamp.',
    ),
    SpecQuote(
        'market-spec.md',
        '§5.2',
        'Iceberg slices receive a new timestamp per slice, **except**: when a single incoming order is matched against multiple iceberg slices, the iceberg is treated as the passive (resting) side throughout and its timestamp does not supersede the incoming order for price determination purposes.',
        note='The iceberg-slice bullet: the iceberg is the passive side for price determination throughout.',
    ),
    SpecQuote(
        'market-spec.md',
        '§5.2',
        'All timestamps across orders and explicit capacity requests are assigned from a single global queue (unique, sequential). Source: Continuous Trading Matching Algorithm §4.3.',
        note="Unique and sequential: a later slice's engine timestamp is strictly later (finding F10).",
    ),
    SpecQuote(
        'market-spec.md',
        '§6.1',
        'A match occurs when best bid price ≥ best ask price.',
        note='The uncrossed-book condition I3 checks after every return.',
    ),
    SpecQuote(
        'market-spec.md',
        '§6.2 item 2',
        'Match is possible if: orders are opposite sides, same contract, and `sell_price ≤ buy_price`.',
        note='The crossing test of the reference walk (W).',
    ),
    SpecQuote(
        'market-spec.md',
        '§6.2 item 3',
        "Trade executes at the **resting order's limit price** (maker pricing). Source: Continuous Trading Matching Algorithm §5.1.1.",
        note="The incoming-iceberg draw: each trade prints at the resting level's limit.",
    ),
    SpecQuote(
        'market-spec.md',
        '§6.2 item 6',
        'If matching exhausts one price level, continues to next best price level.',
        note='The incoming-iceberg draw walks several levels.',
    ),
    SpecQuote(
        'market-spec.md',
        '§6.2 item 7',
        "Iceberg: when a single incoming order matches multiple slices of the same iceberg, the iceberg is always the price-setter. Source: Continuous Trading Matching Algorithm §5.1.2. This includes the §4.2.1 Rule-3 hop, where a regenerated slice crosses the same aggressor's resting remainder (OQ-31, resolved 2026-10-02).",
        note='The resting iceberg is always the price-setter, the Rule-3 hop included.',
    ),
    # --- open questions and the domain review ---
    SpecQuote(
        'open-questions.md',
        'OQ-31',
        "The trade price of the Rule-3 hop is the **iceberg slice's own limit price**: the base price for a standard iceberg (PPD = 0), or the slice's drifted price $\\mathit{base} + k\\,\\mathit{ppd}$ for an ICEwPI slice $k$. It is **not** the aggressor's limit $L$ carried by its resting remainder.",
        block='Resolution (2026-10-02',
        note="OQ-31's ruling (Reading B).",
    ),
    SpecQuote(
        'open-questions.md',
        'OQ-31',
        'When $L$ equals the slice price the readings coincide, so a record builder should draw $L$ strictly through the slice price for the test to discriminate.',
        block='Consequences.',
        note="Why the Rule-3 draw's L is checked to lie strictly through p_ice_int (section 4 precondition).",
    ),
    SpecQuote(
        'open-questions.md',
        'OQ-33',
        '*"The real market treats it as a chain of instances, each the size of the peak. Instance may only trade at its own limit, where is the increment. The sweep stops at the first instance that cannot trade. The leftover keeps its place in the ladder, and trade rows split where one instance ends and the next begins."*',
        block='Revised by user ruling (2026-10-20',
        note="The user's binding ruling of 2026-10-20, byte-copied as the document records it.",
    ),
    SpecQuote(
        'open-questions.md',
        'OQ-33',
        'The incoming iceberg is a chain of instances of size $P$ (the last may be smaller). Each trade is between one resting counter-bid entry and one incoming instance.',
        block='Revised by user ruling (2026-10-20',
        note="OQ-33 as revised: the incoming draw's trade sequence is the (counter bid, instance) partition. (part 1 of 2; the passage is quoted in full, in consecutive parts)",
    ),
    SpecQuote(
        'open-questions.md',
        'OQ-33',
        'A row ends where a resting entry ends **or** where one incoming instance ends and the next begins. Each resting NON order is one counter bid, each instance of a resting iceberg is its own counter bid, and two resting instances are never merged into one row. This holds for standard icebergs ($d = 0$) and for ICEwPI.',
        block='Revised by user ruling (2026-10-20',
        note='(continued, part 2 of 2) It holds at PPD 0, every C08 iceberg.',
    ),
    SpecQuote(
        'open-questions.md',
        'OQ-33',
        'The 2026-10-07 ruling above is superseded for incoming icebergs where it says that an incoming peak boundary never splits a row, and where it calls one-trade-per-incoming-instance unsupported. Its other half stands: no row spans two resting entries.',
        block='Revised by user ruling (2026-10-20',
        note='Why the r3 known result and its quote of the 2026-10-07 ruling are retired (finding F6).',
    ),
    SpecQuote(
        'open-questions.md',
        'OQ-34',
        'A remainder $R > 0$ keeps its place in the ladder: it rests as instance $k_e = \\lfloor M/P \\rfloor$ at $b + k_e\\,d$, and later instances continue from there. The drift does not restart at slice 0.',
        block='Resolved by user ruling (2026-10-20',
        note='OQ-34 resolved: continuation of the ladder, the (R) rest price and instance index.',
    ),
    SpecQuote(
        'open-questions.md',
        'OQ-34',
        'With $u = M - k_e P$, the rested instance shows $v_1 = \\min(P - u, R)$, its unused part, and $R - v_1$ is hidden. This is "at most the reduced part" (Annex §4.4); the sources do not state the display of a part-used instance, so this choice is derived from the data below.',
        block='Resolved by user ruling (2026-10-20',
        note='OQ-34 resolved: the display v_1 of (R), derived from data (findings F14, F16).',
    ),
    SpecQuote(
        'open-questions.md',
        'OQ-20',
        "An active iceberg slot whose `order_id` is absent from the LOB at rest between aggressors — i.e. the visible peak was fully consumed by a prior aggressor's matching process and the next slice has not yet been activated — is a **permitted between-aggressor state** under the matching algorithm.",
        block='Ruling.',
        note='The permitted between-aggressor state: a consumed slice whose successor is not yet shown.',
    ),
    SpecQuote(
        'open-questions.md',
        'OQ-23',
        '**Therefore Q2 = NOT a Rule 4 violation; within tolerance.**',
        block='Q2 — Rule-4 verdict',
        note='The second route to the same stale slot: an engineering invariant, not a Rule-4 breach.',
    ),
    SpecQuote(
        'scenario-ladder-review.md',
        '§17.3',
        "C08's K2 must still follow V16 and OQ-31's Consequences on their own terms. Every `C08.rule3` trade\n  prints at the slice price, and $L$ is drawn strictly through the slice price. That requirement comes",
        note='The D3 gate for this K2.',
    ),
    SpecQuote(
        'scenario-ladder-review.md',
        'D3-08',
        "Any scenario that draws two same-price icebergs against one\naggressor (C05's interloper family, C08's multi-slice draw, C09) should pin the exact counterparty\nsequence in its Known result.",
        note='Why every draw pins the exact counterparty sequence (finding F9).',
    ),
)
CITATIONS = (
    Citation(
        'Continuous Trading Matching Algorithm (public description) §3.3 — docs/omiecid/corpus/Mapping_Authoritative_Documents_EU_Continuous_Intraday_Trading_SIDC_XBID_Spain_OMIE/Continuous_Trading_Matching_Algorithm_Public_Description.md:257-276',
        'Iceberg orders are limit orders which are only visible with part of their total quantity in the market, while their full quantity is exposed to the market for matching, and which may come with a range of limit prices. For each iceberg order, a peak size and a total quantity are defined. An iceberg order is displayed in the market with a quantity equal to its peak size (if the o rder is not matched immediately). The quantity visible in the order book, is called the shown quantity. The shown quantity of an iceberg order is always identical to its peak size when it is newly entered and not matched. When an iceberg order is matched in a trade, its total quantity is reduced by the trade quantity. If the shown quantity before the trade was greater than the subtracted amount, the order remains visible in the market with the remaining shown quantity. If the shown quantity before the trade was less than or equal to the subtracted amount, a new slice of order quantity is made available in the market sized at the peak size of the iceberg order or at the remaining ## Page 9 9 quantity, if this is smaller than the peak size . When the quantity of the last slice has been reduced to zero, the iceberg order is fully executed and removed from the order book. Iceberg orders can be entered with a peak price delta. Each new slice will then be entered',
    ),
    Citation(
        'Continuous Trading Matching Algorithm (public description) §5.1, §5.1.1, §5.1.2 — same file:462-479',
        'Regular matching is triggered by the entry of an order with a new time stamp. An order with a new time stamp may be a newly entered order, a modified order, a (re)activated order that was inactive before, or a new slice of an iceberg order. 5.1.1. Price determination When two orders are matched in a regular match, one of these orders must be an order with a new timestamp and the other one must be an order already present in the order book. The price at which two orders are matched becomes the price of the trade that is concluded. Two orders are matched at the limit price of the order that was already in the order book. If a buy order with a new timestamp is matched against an existing sell order, the limit price of the sell order becomes the trade execution price. If a sell order with a new timestamp is matched against an existing buy order, the limit price of the buy ord er becomes the trade execution price. 5.1.2. Iceberg orders in regular matching In a matching process where a single order with a new timestamp is matched against more than one slice of an iceberg order already in the order book, the price is always determined by the iceberg order already in the order book and never by the order with the new timestamp, even if the timestamp of the iceberg order is renewed during the matching process.',
    ),
    Citation(
        'Continuous Trading Matching Algorithm (public description) §5.1.3 — same file:484-493',
        '5.1.3. Matching against multiple orders If an order with a new timestamp can be executed, it is not necessarily executed at a single price (except orders with the execution restriction AON ), but may sequentially generate multiple transactions at different prices against multiple different orders that already existed in the order book. As soon as the order has been executed against all orders at a certain price limit, the next best price level becomes best and the order continuous to be matched against orders entered at this price level. This process continuous as long as the incoming order remains executable and has a positive order quantity. Subsequently , the order is deleted if the order quantity has reached zero or if it has the IOC execution restriction. In all other cases, the order is entered into the order book with its remaining quantity.',
    ),
    Citation(
        'OMIE Rules, Iceberg bids §4.4 — docs/omiecid/corpus/Mapping_Authoritative_Documents_EU_Continuous_Intraday_Trading_SIDC_XBID_Spain_OMIE/OMIE_Day-Ahead_and_Intraday_Electricity_Market_Rules_English.md:10262-10267',
        '- If, while there is an “Iceberg” bid in the Bid Book, a counter bid with a competitive price at a quantity higher than the viewable quantity of the “Iceberg” bid is entered, different transactions will be performed for each instance of the “Iceberg” bid, each with its own creation time. The viewable amount of the “Iceberg” bid in the Bid Book will be the amount that has remained unmatched after the last instance.',
    ),
    Citation(
        'OMIE Rules, Iceberg bids §4.4 (incoming iceberg) — docs/omiecid/corpus/Mapping_Authoritative_Documents_EU_Continuous_Intraday_Trading_SIDC_XBID_Spain_OMIE/OMIE_Day-Ahead_and_Intraday_Electricity_Market_Rules_English.md:10256-10261',
        '- When entering an “Iceberg” bid with a competitive price, the quantity to be considered will be the total quantity of the “Iceberg” bid, recording a transaction for each counter bid that it matches with. Additionally, if the total quantity does not match, the viewable quantity that will appear in the Bid Book will be at most the reduced part specified when creating the bid, even if the matched quantity was not a multiple of the reduced part.',
    ),
    Citation(
        'Continuous Trading Matching Algorithm (public description) §5.3 — same file as §3.3:576-590',
        'Whenever two orders are matched, a trade is created. Even though one order can match with multiple other or ders, a trade is always between exactly two orders. […] The timestamp of a trade reflects the moment that one of the orders that will enter into the trade is found to be executable (against the other order) by the algorithm.',
    ),
)

CITATIONS = CITATIONS + (
    Citation(
        'OMIE Rules, ICEBERG bids with incremental pricing §4.5 — docs/omiecid/corpus/Mapping_Authoritative_Documents_EU_Continuous_Intraday_Trading_SIDC_XBID_Spain_OMIE/OMIE_Day-Ahead_and_Intraday_Electricity_Market_Rules_English.md:10276-10278',
        'As an option, “Iceberg” orders can be entered with a price increment. If entered, each new instance will have a new price, which will be calculated as the price of the previous instance plus the price increment.',
    ),
)

QUOTE_CHECKS = check_quotes(QUOTES)
display(Markdown(render_quotes(QUOTE_CHECKS, CITATIONS)))
print(f"{sum(c.ok for c in QUOTE_CHECKS)} of {len(QUOTE_CHECKS)} quotes match their source")

## 3. Configuration

One visible parameter cell. The headless gate overrides it by injecting a cell right after it (`execute_notebook(..., parameters={...})`); everything derived from it is computed in the next cell.

- **`DRAWS = 500`** valid draws of `C08.iceberg_cascade` from `SEED0` (the card's default for a random scenario), **`RULE3_DRAWS = 250`** valid draws of `C08.rule3` from `RULE3_SEED0`, **`INCOMING_DRAWS = 250`** valid draws of `C08.incoming_iceberg` from `INCOMING_SEED0`, and **`REMAINDER_DRAWS = 250`** valid draws of `C08.incoming_remainder` from `REMAINDER_SEED0`: 1,250 runs in all. The four seed ranges are disjoint, so one seed names one draw.
- **Discards.** A seed the generator refuses (`CapacityBreach`), or a tape outside the capacity margin before the run (including an analytic hop bound above the 100-pass bound, finding F11), or a run that the battery's I8 marks `invalid` after it, is **discarded and replaced by the next seed** of its range. Each construction stops at its quota of valid draws or after twice that many attempts; the discard rate is reported. An I8 discard of a draw that passed the pre-run margin is also a failed assertion (finding F7).
- **`EAGER_DRAWS = 3`.** The first `EAGER_DRAWS` **valid** draws of each construction (the first valid seeds, finding F12) also run eager, and each eager record is compared, integer field by integer field, with its jitted record (`compare_records`). On top of these, an eager run is **forced** for the first valid draw of each of the fourteen classes below. Every class **must** have its eager comparison: a missing one is a failed `eager_jit` assertion. Eager runs are chosen among valid draws, so a discarded seed cannot remove them.
- **The fourteen classes** (each must also appear among the valid draws: the coverage assertion of section 7): cascade × iceberg side × ($r < peak$ or $r = peak$, every slice full); Rule-3 × iceberg side × ($q_{agg} \bmod peak > 0$ or $= 0$, finding F4); incoming × iceberg side × (one level or several levels walked); remainder × iceberg side. The default seed ranges hold at least 3 draws of every class. Separately, the valid incoming draws must include at least one draw whose rows split one level's fill at an incoming instance boundary (finding F16); the default range holds many. The valid incoming-remainder draws must include at least one with $R > 0$ and $u > 0$ (finding F14).
- The tick grids are **not** parameters: they come from `OMIEProductConfig()` and serve only to display euros and to convert the engine's float report to ticks.

A run with far fewer draws than the default can miss a class; it then fails the coverage check and that class's eager comparison. That is intended: such a run does not cover the card's draws.

In [ ]:
CERT_ID = "C08"            # C00-C17 are the ladder; C90-C99 are reserved for the harness
DRAWS = 500                # valid draws of C08.iceberg_cascade (card: random scenarios default to 500 draws)
RULE3_DRAWS = 250          # valid draws of C08.rule3 (the card's separate Rule-3 draw)
INCOMING_DRAWS = 250       # valid draws of C08.incoming_iceberg (the card's separate incoming-iceberg draw)
SEED0 = 0                  # first seed of C08.iceberg_cascade
RULE3_SEED0 = 1_000_000    # first seed of C08.rule3 (a disjoint range: one seed names one draw)
INCOMING_SEED0 = 2_000_000 # first seed of C08.incoming_iceberg (a disjoint range)
REMAINDER_DRAWS = 250      # valid draws of C08.incoming_remainder (the incoming iceberg that leaves a remainder)
REMAINDER_SEED0 = 3_000_000  # first seed of C08.incoming_remainder (a disjoint range)
EAGER_DRAWS = 3            # first EAGER_DRAWS valid draws of each construction also run eager (plus forced classes)

In [ ]:
import pathlib

import numpy as np

from omiecid_simulator.jaxomiecid import omie_tape_synth as synth
from omiecid_simulator.omiecid.omie_config import OMIEProductConfig
from omiecid_simulator.omiecid.order_types import ExecRestriction, Side, Validity

CASCADE, RULE3, INCOMING = "C08.iceberg_cascade", "C08.rule3", "C08.incoming_iceberg"
REMAINDER = "C08.incoming_remainder"   # make_iceberg_cascade(seed, variant="incoming_remainder"); not in CONSTRUCTIONS
SHORT = {CASCADE: "cascade", RULE3: "rule3", INCOMING: "incoming", REMAINDER: "remainder"}
INCOMING_LIKE = (INCOMING, REMAINDER)   # an incoming iceberg against resting NON levels
PLAN = (   # (construction, valid draws, first seed); each stops after 2 x its draws attempts
    (CASCADE, DRAWS, SEED0),
    (RULE3, RULE3_DRAWS, RULE3_SEED0),
    (INCOMING, INCOMING_DRAWS, INCOMING_SEED0),
    (REMAINDER, REMAINDER_DRAWS, REMAINDER_SEED0),
)
TOTAL_DRAWS = sum(n for _, n, _ in PLAN)

CONFIG = OMIEProductConfig()
ENVELOPE = synth.DEFAULT_ENVELOPE
BUY, SELL = int(Side.BUY), int(Side.SELL)
NON, IOC, FOK, ICE = (int(ExecRestriction.NON), int(ExecRestriction.IOC), int(ExecRestriction.FOK),
                      int(ExecRestriction.ICE))
EXEC_NAME = {NON: "NON", IOC: "IOC", FOK: "FOK", ICE: "ICE"}
GFS = int(Validity.GFS)

SLICE_MARGIN = 64                     # capacity-validity precondition: every iceberg has ceil(total/peak) <= 64
PASS_BOUND = 100                      # the iteration ceiling: 100 outer matching passes (_TRIP_BOUND; ladder v2.4, S7)
PRICE_INT_MAX = 1_999_800             # §2.3: +9,999 EUR/MWh in the offset encoding (0 is -9,999)
QTY_INT_MIN, QTY_INT_MAX = 1, 9_990   # §2.3: 0.1 MW and 999 MW in quantity ticks

CLASSES = (
    [f"cascade, iceberg {s}, {r}" for s in ("BUY", "SELL") for r in ("r < peak", "r = peak")]
    + [f"rule3, iceberg {s}, {r}" for s in ("BUY", "SELL") for r in ("q_agg mod peak > 0", "q_agg mod peak = 0")]
    + [f"incoming, iceberg {s}, {r}" for s in ("BUY", "SELL") for r in ("one level", "several levels")]
    + [f"remainder, iceberg {s}" for s in ("BUY", "SELL")]
)

_own = sorted(pathlib.Path.cwd().glob(f"{CERT_ID}_*.ipynb"))
NOTEBOOK = _own[0].name if len(_own) == 1 else "C08_iceberg_regeneration.ipynb"

for _c, _n, _s0 in PLAN:
    print(f"{CERT_ID}: {_c} x {_n} valid draws from seed {_s0} (at most {2 * _n} attempts)")
print(f"eager: first {EAGER_DRAWS} valid draws of each construction + the first of each of {len(CLASSES)} classes")
print(f"price tick {CONFIG.price_tick} EUR/MWh, qty tick {CONFIG.qty_tick} MW (display and report conversion only)")
print(f"capacity envelope: {ENVELOPE}; slice margin <= {SLICE_MARGIN} per iceberg; hop bound <= {PASS_BOUND} passes")

## 4. Dataset

Each construction emits a short GFS tape on one contract: order ids $1, 2, \ldots$ and tape timestamps $1, 2, \ldots$ in row order, real rows first, then pad rows. Write $\sigma$ for the iceberg's side and $\bar\sigma$ for the other.

| construction | rows (submission $s$) | iceberg | the other order(s) |
|---|---|---|---|
| `C08.iceberg_cascade` | iceberg $s = 0$, aggressor $s = 1$ | $\sigma$ ICE at $p_{ice}$ (30–90 EUR/MWh), PPD 0, $peak$ 1–10 MW, $n = \lceil total/peak \rceil \in [2, 39]$, last slice $r = total - (n-1)\,peak \in [1, peak]$ | $\bar\sigma$ NON, limit $p_{ice} \pm 2$ EUR through, size **exactly** $total$ |
| `C08.rule3` | iceberg $s = 0$, aggressor $s = 1$ | $\sigma$ ICE at $p_{ice}$, PPD 0, $peak$ 1–10 MW, $n \in [3, 39]$ | $\bar\sigma$ NON, limit $L$ 1–200 ticks **strictly** through $p_{ice}$, size $peak < q_{agg} \le total - peak$ |
| `C08.incoming_iceberg` | levels $s = 0 \ldots L-1$, iceberg $s = L$ | $\sigma$ ICE at $p_{ice}$ = the deepest level it needs, PPD 0, $peak$ 1–6 MW, $n \in [2, 30]$ | $L \in [2, 8]$ resting $\bar\sigma$ NON levels (each one counter bid), prices worsening by a fixed 1–50 ticks, total depth $\ge total + 1$ tick; so the sweep takes $M = total$ and leaves $R = 0$ (finding F14) |
| `C08.incoming_remainder` | levels $s = 0 \ldots L-1$, iceberg $s = L$ | $\sigma$ ICE at $p_{ice}$ = the deepest level, PPD 0, $peak$ 1–6 MW | $L \in [2, 8]$ resting $\bar\sigma$ NON levels at distinct prices, every one crossing $p_{ice}$, total depth $M < total$ with $M \bmod peak \ne 0$; so $R = total - M > 0$ and $u = M - \lfloor M/peak \rfloor\,peak > 0$ (finding F14) |

**What section 5 reads.** The expectation reads the **tape rows** through one reference walk, and `draw.params` for the closed form of each construction, which is cross-checked against the walk on every draw and against the tape (the precondition below). `draw.expected` is never read, and the generator's `pending` tag on `C08.rule3` (OQ-31) is provenance only.

**The precondition, checked from the tape for every valid draw** (an aggregate assertion in section 7): the row count, sides, restrictions (ICE / NON), validity GFS, no price increment (PPD 0) on any row, a peak only on the iceberg, order ids and tape timestamps strictly increasing, prices and sizes within §2.3, every value equal to its `params` entry, and the construction's own shape: cascade ($q_{agg} = total$, the limit crosses); Rule-3 ($n \ge 3$, $peak < q_{agg} \le total - peak$, $L$ crosses $p_{ice}$ and $L \ne p_{ice}$); incoming (level prices pairwise distinct and strictly worsening, $p_{ice}$ one of them, the levels that cross $p_{ice}$ hold at least $total$); incoming-remainder (the same level shape, every level crosses $p_{ice}$, their total depth $M$ is below $total$, $M \bmod peak \ne 0$, and `params["ppd_int"]` $= 0$).

**The capacity margin** (scenario-ladder-plan, *The capacity-validity precondition*, five ceilings), checked from the tape **before** each run: the draw's analytic bounds (`draw.capacity_bounds`, and the generic `tape_capacity_bounds(tape)`) within the envelope (≤ 80 resting per side, ≤ 80 trades, hop bound ≤ 160, ≤ 800 concurrent icebergs) **and** both hop bounds ≤ 100, the real iteration ceiling of 100 outer matching passes (the envelope's 160 does not bound the run; finding F11), and the largest $\lceil qty/peak \rceil$ over the tape's iceberg rows ≤ 64. A tape outside it is never run. After the run, the battery's I8 confirms the flags.

Below: the first draw of each construction in readable units beside the integers, the book shapes, and the class counts of the initial seed ranges.

In [ ]:
from collections import Counter

from omiecid_simulator.omiecid.encoding import decode_price, decode_qty

C_CONTRACT, C_SIDE, C_PRICE, C_QTY, C_OID = (synth.COL_CONTRACT, synth.COL_SIDE, synth.COL_PRICE, synth.COL_QTY,
                                             synth.COL_OID)
C_AGENT, C_TS, C_EXEC, C_VALIDITY, C_PEAK, C_PPD = (synth.COL_AGENT, synth.COL_TS, synth.COL_EXEC,
                                                    synth.COL_VALIDITY, synth.COL_PEAK, synth.COL_PPD)


def real_rows(tape):
    # The real rows (order_id >= 1), in tape (= submission) order.
    t = np.asarray(tape)
    return t[t[:, C_OID] >= 1]


def side_name(side):
    return "BUY" if int(side) == BUY else "SELL"


def other(side):
    return SELL if int(side) == BUY else BUY


def crosses(rest_side, p_rest, p_in):
    # §6.1 / §6.2 item 2: sell price <= buy price, for a resting order on rest_side and an incoming limit p_in.
    return int(p_rest) <= int(p_in) if int(rest_side) == SELL else int(p_rest) >= int(p_in)


def ceil_div(a, b):
    return -(-int(a) // int(b))


def max_slices(tape):
    # Largest ceil(qty / peak) over the tape's iceberg rows (0 when there are none).
    ice = [r for r in real_rows(tape) if int(r[C_EXEC]) == ICE]
    return max((ceil_div(r[C_QTY], max(1, int(r[C_PEAK]))) for r in ice), default=0)


DRAWN = {}       # (construction, seed) -> Draw, or None when the generator refused the seed
REFUSED = {}     # (construction, seed) -> the generator's CapacityBreach message


def prepare(construction, seed):
    # Draw one seed (the run loop of section 7 calls this for a re-draw too).
    try:
        d = (synth.make_iceberg_cascade(seed, variant="incoming_remainder") if construction == REMAINDER
             else synth.CONSTRUCTIONS[construction](seed))
    except synth.CapacityBreach as exc:
        d = None
        REFUSED[(construction, seed)] = str(exc)
    DRAWN[(construction, seed)] = d
    return d


for _c, _n, _s0 in PLAN:
    for _s in range(_s0, _s0 + _n):
        prepare(_c, _s)


def tape_table(draw, title):
    rows = ["| s | oid | side | exec | price (EUR/MWh) | qty (MW) | peak (MW) | price_int | qty_int | peak_int | "
            "agent | tape ts |", "|---|---|---|---|---|---|---|---|---|---|---|---|"]
    for k, r in enumerate(real_rows(draw.tape)):
        rows.append(f"| {k} | {int(r[C_OID])} | {side_name(r[C_SIDE])} | {EXEC_NAME.get(int(r[C_EXEC]), int(r[C_EXEC]))} | "
                    f"{decode_price(int(r[C_PRICE]), CONFIG):.2f} | {decode_qty(int(r[C_QTY]), CONFIG):.1f} | "
                    f"{decode_qty(int(r[C_PEAK]), CONFIG):.1f} | {int(r[C_PRICE])} | {int(r[C_QTY])} | "
                    f"{int(r[C_PEAK])} | {int(r[C_AGENT])} | {int(r[C_TS])} |")
    return f"**{title}**\n\n" + "\n".join(rows)


FIRST = {}
for _c, _n, _s0 in PLAN:
    _s = next((s for s in range(_s0, _s0 + _n) if DRAWN[(_c, s)] is not None), None)
    FIRST[_c] = _s
    if _s is None:
        print(f"{_c}: every seed of the initial range was refused by the generator")
        continue
    _d = DRAWN[(_c, _s)]
    display(Markdown(tape_table(_d, f"{_c}: seed {_s} (digest `{_d.digest()}`)")))
    print(f"seed {_s} params: {_d.params}")
    print(f"seed {_s} capacity bounds: {dict(_d.capacity_bounds)}; largest ceil(qty/peak): {max_slices(_d.tape)}")
print(f"seeds refused by the generator in the initial ranges: {len(REFUSED)}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(18, 3.6))
for ax, (_c, _n, _s0) in zip(axes, PLAN):
    if FIRST.get(_c) is None:
        ax.set_title(f"{SHORT[_c]}: no draw")
        continue
    _d = DRAWN[(_c, FIRST[_c])]
    for r in real_rows(_d.tape):
        buy = int(r[C_SIDE]) == BUY
        sgn = 1 if buy else -1
        p = decode_price(int(r[C_PRICE]), CONFIG)
        q = decode_qty(int(r[C_QTY]), CONFIG)
        col = "tab:green" if buy else "tab:red"
        if int(r[C_EXEC]) == ICE:
            pk = decode_qty(int(r[C_PEAK]), CONFIG)
            ax.barh(p, sgn * pk, height=0.5, color=col, label=f"iceberg {side_name(r[C_SIDE])}: visible peak {pk:.1f} MW")
            ax.barh(p, sgn * (q - pk), left=sgn * pk, height=0.5, color=col, alpha=0.25, hatch="//",
                    label=f"hidden {q - pk:.1f} MW (total {q:.1f})")
        else:
            ax.barh(p, sgn * q, height=0.35, color=col, alpha=0.6,
                    label=None if _c in INCOMING_LIKE and int(r[C_OID]) > 1 else f"{side_name(r[C_SIDE])} NON")
    ax.axvline(0, color="black", lw=0.8)
    ax.set_xlabel("quantity (MW): buy right, sell left")
    ax.set_ylabel("limit (EUR/MWh)")
    ax.set_title(f"{SHORT[_c]}, seed {FIRST[_c]}", fontsize=9)
    ax.legend(loc="best", fontsize=6)
plt.tight_layout()
plt.show()


def class_of(construction, tape):
    # The class of a draw, read from its tape (section 3).
    rows = real_rows(tape)
    ice = [r for r in rows if int(r[C_EXEC]) == ICE][0]
    side = side_name(ice[C_SIDE])
    total, peak = int(ice[C_QTY]), int(ice[C_PEAK])
    if construction == CASCADE:
        n = ceil_div(total, peak)
        return f"cascade, iceberg {side}, " + ("r = peak" if total - (n - 1) * peak == peak else "r < peak")
    if construction == RULE3:
        q_agg = int([r for r in rows if int(r[C_EXEC]) != ICE][0][C_QTY])
        return f"rule3, iceberg {side}, " + ("q_agg mod peak = 0" if q_agg % peak == 0 else "q_agg mod peak > 0")
    if construction == REMAINDER:
        return f"remainder, iceberg {side}"
    levels = [r for r in rows if int(r[C_EXEC]) != ICE]
    left, used = total, 0
    for lv in sorted(levels, key=lambda r: rank_key(r[C_SIDE], r[C_PRICE], 0)):
        if left <= 0 or not crosses(lv[C_SIDE], lv[C_PRICE], ice[C_PRICE]):
            break
        left -= int(lv[C_QTY])
        used += 1
    return f"incoming, iceberg {side}, " + ("one level" if used <= 1 else "several levels")


def rank_key(side, price, entry):
    # §6.1: SELL lowest price first, BUY highest price first; ties by the earlier entry (timestamp).
    return ((int(price) if int(side) == SELL else -int(price)), int(entry))


_classes = Counter(class_of(c, d.tape) for (c, s), d in DRAWN.items() if d is not None)
display(Markdown("**Classes in the initial seed ranges** (counts)\n\n| class | draws |\n|---|---|\n"
                 + "\n".join(f"| {k} | {_classes.get(k, 0)} |" for k in CLASSES)))

## 5. Known result

**Notation.** Integer ticks throughout. An iceberg has limit (base) price $p_{ice}$, price increment $ppd = 0$ (every C08 tape), total $total$ and peak $peak$; its slices are $k = 0, 1, \ldots, n-1$ with $n = \lceil total/peak \rceil$, slice $k$ priced $p_{ice} + k\,ppd = p_{ice}$ and sized $peak$ except the last, $r = total - (n-1)\,peak$ (§4.2; §4.2.1 Rule 2's worked example 3, 3, 3, 1). A trade row is pinned as $[\,buyer,\ seller,\ price,\ qty,\ s\,]$: the buyer's and seller's order ids, the price and quantity in ticks, and the submission $s$ that wrote it (finding F3).

**(W) The reference walk** (§4.2, §4.2.1 Rules 2–4, §5.2, §6.1, §6.2). The real rows are processed in submission order; each entry gets the next number of one global sequence when it enters, and an iceberg slice gets a new one when it is activated (§5.2, "unique, sequential"). An incoming order meets the opposite side ranked by price, then sequence (§6.1); an incoming iceberg aggresses with its **total** quantity (§4.2.1 Rule 2, *Incoming iceberg* (1)) as a chain of instances $k = 0, 1, \ldots$ of size $peak$, the last smaller (item (2)). While it has quantity and the best opposite entry crosses (§6.2 item 2) the limit of its current instance — $b + k\,ppd$ for an incoming iceberg with $k = \lfloor \text{swept}/peak \rfloor$, which is $b$ at PPD 0 (item (6)); the order's limit for a NON — it trades $f = \min(\text{left},\ \text{visible},\ \text{instance left})$, one row per (resting entry, incoming instance) pair (item (2); OQ-33 revised 2026-10-20), where *instance left* is what remains of the incoming iceberg's current instance ($\infty$ for a NON), **at the resting entry's price** (§6.2 item 3; for a resting iceberg, its slice price: §6.2 item 7, §5.2, and at the Rule-3 configuration OQ-31). A resting slice used up is replaced at once by the iceberg's next slice, a new entry with a new sequence number, re-ranked among the counter bids (Rule 2, *Re-ranking*), which the same incoming order keeps matching (Rule 2; Rule 4: the cascade ends inside this submission). A row therefore ends where a resting entry ends **or** where an incoming instance ends (finding F6). The sweep stops at the first instance that cannot trade (item (6)). Whatever is left at the end rests: a NON as it is; an ICE remainder $R$, after sweeping $M$, continues as instance $k_e = \lfloor M/peak \rfloor$ at $b + k_e\,ppd$, showing $v_1 = \min(peak - u, R)$ with $u = M - k_e\,peak$, and keeps $R - v_1$ hidden (item (4); OQ-34 resolution; not reached by any C08 draw, finding F14). Because the walk is the real algorithm (the incoming order is matched while executable), the Rule-3 decomposition is not modelled separately: it has the same outcome (finding F5).

**Closed forms** (from `draw.params`; checked against (W) on every draw).

*Cascade* (`C08.iceberg_cascade`): $q_{agg} = total$, so the aggressor takes every slice:

$$\text{trades} = \big[\,[b,\ s,\ p_{ice},\ x_k,\ 1]\ \big]_{k=0}^{n-1},\qquad (x_0, \ldots, x_{n-1}) = (\underbrace{peak, \ldots, peak}_{n-1},\ r)$$

with $b, s$ the buyer and seller among $o_{ice}, o_{agg}$; final book empty; filled $= total$ for both; remaining 0 for both.

*Rule-3* (`C08.rule3`): with $q_{agg} = m\,peak + \rho$, $0 \le \rho < peak$, and $m \ge 1$ since $q_{agg} > peak$:

$$\text{trades} = \big[\,[b,\ s,\ p_{ice},\ x_k,\ 1]\ \big]_{k},\qquad (x_k) = (\underbrace{peak, \ldots, peak}_{m},\ \rho \text{ if } \rho > 0)$$

every trade at $p_{ice}$ (**the Rule-3 trade price**, derived from `params["p_ice_int"]` and the tape's $ppd = 0$: slice $k$ is priced $p_{ice} + k \cdot 0$), never at $L$ = `params["p_agg_int"]` $\ne p_{ice}$. The aggressor is filled whole and rests nothing; the iceberg is filled $q_{agg}$ and keeps $total - q_{agg} \ge peak$. Final book: if $\rho > 0$, exactly $\sigma: [[p_{ice},\ peak - \rho,\ o_{ice}]]$ (the slice the aggressor stopped in); if $\rho = 0$, **one of** $\sigma: [[p_{ice},\ \min(peak,\ total - q_{agg}),\ o_{ice}]]$ or $\sigma: []$ (finding F4). **(N)** If the iceberg has a visible row at the end, its engine timestamp is strictly later than its first slice's (finding F10): the observed value is `later`, or `absent` where F4 allows no visible row.

*Incoming* (`C08.incoming_iceberg`): the levels that cross $p_{ice}$, in priority order, with sizes $(\ell_0, \ell_1, \ldots)$, prices $(\pi_0, \pi_1, \ldots)$ and order ids $(o_0, o_1, \ldots)$. The incoming iceberg's instance partition is $(x_0, \ldots, x_{n-1}) = (peak, \ldots, peak, r)$, every instance at limit $p_{ice} + k \cdot 0 = p_{ice}$ (item (6)), which crosses every level the sweep needs. The trades are the **merge** of the two partitions (item (2); OQ-33 revised 2026-10-20). Walk both in order: each row is

$$[\,b,\ s,\ \pi_j,\ t,\ L\,],\qquad t = \min(\text{left of level } j,\ \text{left of instance } k),\qquad \text{price-setter (passive) } = o_j$$

and it closes level $j$, instance $k$, or both; the walk stops when $total$ is filled ($M = total$). So every row is at most $peak$, and the cumulative row sums include every multiple $j\,peak < M$ (the explicit check **(I)**). With $J$ the last level touched and $c$ the number of instance boundaries $j\,peak < M$ that coincide with a level boundary, there are $J + n - c$ rows. On a draw where some instance boundary falls strictly inside a level, a level's fill is split into several rows; the superseded reading (one row per level) is false there (finding F16). Final book: $\bar\sigma$ holds every level with what is left of it (level $J$ keeps its unfilled part if positive, deeper levels are untouched); $\sigma$ is empty, since $R = 0$. **(R)**: no rested remainder (the empty value). The iceberg is filled $total$ and keeps nothing.

*Incoming remainder* (`C08.incoming_remainder`): the same merge, over **every** level (each crosses $p_{ice}$), until the levels are exhausted: $M = \sum_j \ell_j < total$. With $k_e = \lfloor M/peak \rfloor$, $u = M - k_e\,peak$ ($0 < u < peak$) and $R = total - M > 0$, the instance partition actually swept is $(peak, \ldots, peak, u)$ ($k_e$ full instances, then the used part of instance $k_e$), and the rows are its merge with the level partition: every row at most $peak$, every multiple $j\,peak < M$ a cumulative row boundary (I). Final book: $\bar\sigma$ is empty (every level filled). $\sigma$ holds the incoming iceberg's remainder as instance $k_e$ at $p_{ice} + k_e \cdot 0 = p_{ice}$, visible $v_1 = \min(peak - u, R)$ (item (4); OQ-34 *Display C3*). **(R)** $= [p_{ice},\ v_1,\ R - v_1]$. The iceberg is filled $M$ and keeps $R$ (visible + hidden).

**(E) The report** (every construction): for each submission $s$, `[accepted, traded, quantity]` $= [\text{True},\ Q_s > 0,\ Q_s]$, where $Q_s$ is the expected quantity traded in $s$ (0 for every resting submission; $total$, $q_{agg}$ or $total$ for the aggressor or the incoming iceberg); and the trade rows each submission wrote sum to $Q_s$ (**ENGINEERING INVARIANT**, I6(b)). On the cascade and Rule-3 aggressor, the reported price in ticks lies in $[p_{ice} - 1,\ p_{ice} + 1]$ (**ENGINEERING INVARIANT**, finding F8).

**What is asserted, per valid draw** (one combined assertion; expected and observed recorded in full as canonical compact JSON): (A) accepted ids = every order id, rejected = none; (Q)+(S) the trade rows in order, field for field as above — on the cascade and Rule-3 draws their sizes are the slice-size sequence; on the incoming draw, one row per (counter bid, incoming instance) pair, with the passive id per row, plus (I) the instance-split check (`rows_over_peak` and `missing_boundaries`, both expected empty) and (R) the rested remainder (expected empty on every C08 draw); (F) filled and remaining (visible + hidden) per order id, and no order evicted or cancelled; (B) the final book `[price_int, qty_int, oid]` per side, equal to the listed book or one of the listed books; (N) on the Rule-3 draw; (E) as above. Separately: the dataset precondition, the closed form against (W), the battery, I3 and I6 on every valid draw, the capacity checks and the eager comparisons (section 7).

The code below implements (W) as one function of a tape's rows, and the closed forms as one function of `draw.params` per construction. Neither reads `draw.expected`, and neither runs the engine. The closed form of the incoming draw also implements item (4) (the remainder as instance $k_e$ showing $v_1$), and the same branch serves the incoming-remainder draw.

In [ ]:
import json


def canon(x):
    # Canonical compact JSON: how every per-draw expected / observed value is recorded (in full).
    return json.dumps(x, sort_keys=True, separators=(",", ":"))


def walk(tape):
    # (W): the reference walk over the real rows. Returns trades, the final book and per-oid totals.
    book, trades, seq = [], [], [0]

    def nxt():
        seq[0] += 1
        return seq[0]

    def activate(e):
        # The iceberg's next slice: a new entry with a new sequence number (§4.2, §5.2).
        ic = e["ice"]
        ic["k"] += 1
        e["vis"] = min(ic["peak"], e["hidden"])
        e["hidden"] -= e["vis"]
        e["price"] = ic["base"] + ic["k"] * ic["ppd"]
        e["seq"] = nxt()

    for s, r in enumerate(real_rows(tape)):
        side, p, q, oid, ex = int(r[C_SIDE]), int(r[C_PRICE]), int(r[C_QTY]), int(r[C_OID]), int(r[C_EXEC])
        ice = ex == ICE
        peak = int(r[C_PEAK]) if ice else q
        ppd = int(r[C_PPD])
        inc = {"side": side, "price": p, "vis": 0, "hidden": 0, "oid": oid, "seq": nxt(), "fresh": False,
               "ice": {"base": p, "ppd": ppd, "peak": peak, "k": 0} if ice else None}
        left = q        # an incoming iceberg aggresses with its TOTAL (Rule 2, Incoming iceberg (1)); a NON with its qty
        swept = 0       # M so far
        while left > 0:
            k = swept // peak if ice else 0     # the incoming instance now trading (item (2): a chain of size-peak instances)
            lim = p + k * ppd if ice else p     # its own limit b + k d (item (6)); PPD 0: every instance at b
            opp = [e for e in book if e["side"] != side and e["vis"] > 0 and crosses(e["side"], e["price"], lim)]
            if not opp:
                break                           # the first instance that cannot trade ends the sweep (item (6))
            best = min(opp, key=lambda e: rank_key(e["side"], e["price"], e["seq"]))   # re-ranked every time
            inst_left = peak - swept % peak if ice else left    # what is left of the incoming instance
            f = min(left, best["vis"], inst_left)   # a row ends at a resting entry's end OR an incoming instance's end
            buyer, seller = (oid, best["oid"]) if side == BUY else (best["oid"], oid)
            trades.append({"row": [buyer, seller, best["price"], f, s], "passive": best["oid"]})
            left -= f
            swept += f
            best["vis"] -= f
            best["fresh"] = False
            if best["vis"] == 0:
                if best["hidden"] > 0:
                    activate(best)              # Rule 2 / Rule 4: the next slice, matched in this same submission
                    best["fresh"] = True        # untouched since activation (finding F4)
                else:
                    book.remove(best)
        if left > 0 and ex in (NON, ICE):       # §6.2 item 4 / §4.5: what is left rests
            if ice:
                # item (4); OQ-34: the remainder R continues as instance k_e = floor(M/P) at b + k_e d,
                # showing v_1 = min(P - u, R), u = M - k_e P, and keeps R - v_1 hidden
                k_e = swept // peak
                u = swept - k_e * peak
                inc["ice"]["k"] = k_e
                inc["price"] = p + k_e * ppd
                inc["vis"] = min(peak - u, left)
            else:
                inc["vis"] = left               # a NON shows all of it
            inc["hidden"] = left - inc["vis"]
            book.append(inc)
    return trades, book


def instance_split(qtys, peak):
    # (I), item (2) on one submission's rows in row order: the rows above peak, and the multiples j*peak < M
    # (M = the rows' sum) that are not a cumulative row boundary. Both empty when the rows split at instances.
    qs = [int(x) for x in qtys]
    cum, acc = set(), 0
    for x in qs:
        acc += x
        cum.add(acc)
    return {"rows_over_peak": [x for x in qs if x > int(peak)],
            "missing_boundaries": [j * int(peak) for j in range(1, ceil_div(acc, peak)) if j * int(peak) not in cum]}


def rest_of(entries, oid):
    # (R): the rested remainder of `oid` as [price, visible, hidden], or [] when nothing of it rests.
    rows = [e for e in entries if e["oid"] == oid and e["vis"] + e["hidden"] > 0]
    return [rows[0]["price"], rows[0]["vis"], rows[0]["hidden"]] if rows else []


def book_view(entries, skip=()):
    out = {"BUY": [], "SELL": []}
    for e in entries:
        if e["vis"] > 0 and e["oid"] not in skip:
            out[side_name(e["side"])].append([e["price"], e["vis"], e["oid"]])
    return {k: sorted(v) for k, v in out.items()}


def expectation(construction, draw):
    # Everything section 7 compares for one draw, from (W) over the tape's rows.
    rows = real_rows(draw.tape)
    oids = [int(r[C_OID]) for r in rows]
    trades, book = walk(draw.tape)
    filled = {o: 0 for o in oids}
    for t in trades:
        filled[t["row"][0]] += t["row"][3]
        filled[t["row"][1]] += t["row"][3]
    remaining = {o: 0 for o in oids}
    for e in book:
        remaining[e["oid"]] += e["vis"] + e["hidden"]
    q_by_sub = [sum(t["row"][3] for t in trades if t["row"][4] == s) for s in range(len(rows))]
    fresh = [e["oid"] for e in book if e["fresh"]]
    books = [book_view(book)]
    if fresh:                                   # finding F4: a just-activated, untouched slice may be absent
        books.append(book_view(book, skip=set(fresh)))
    ice_row = [r for r in rows if int(r[C_EXEC]) == ICE][0]
    ice_oid, ice_sub = int(ice_row[C_OID]), oids.index(int(ice_row[C_OID]))
    exact = {
        "accepted": sorted(oids), "rejected": [],
        "trades": [t["row"] for t in trades],
        "filled": {str(o): filled[o] for o in oids},
        "remaining": {str(o): remaining[o] for o in oids},
        "lost": {},
        "reports": [[True, q > 0, q] for q in q_by_sub],
        "rows_qty": q_by_sub,
    }
    allowed = {"final_book": books}
    walk_split = None
    if construction in INCOMING_LIKE:
        exact["passive"] = [t["passive"] for t in trades]      # finding F3: the resting level is the price-setter
        exact["instance_split"] = {"rows_over_peak": [], "missing_boundaries": []}   # (I): item (2), by rule
        exact["incoming_rest"] = rest_of(book, ice_oid)          # (R): item (4); [] when R = 0
        walk_split = instance_split([t["row"][3] for t in trades if t["row"][4] == ice_sub], int(ice_row[C_PEAK]))
    if construction == RULE3:
        ice_left = [e for e in book if e["oid"] == ice_oid and e["vis"] > 0]
        allowed["new_slice_ts"] = ["later", "absent"] if fresh else (["later"] if ice_left else ["absent"])
    agg_sub = 1 if construction in (CASCADE, RULE3) else None
    if agg_sub is not None:
        exact["report_price_in_bracket"] = True
    return {"exact": exact, "allowed": allowed, "oids": oids, "ice_oid": ice_oid, "ice_sub": ice_sub,
            "ice_side": int(ice_row[C_SIDE]), "agg_sub": agg_sub, "p_ice": int(ice_row[C_PRICE]),
            "n_slices": ceil_div(ice_row[C_QTY], ice_row[C_PEAK]), "class": class_of(construction, draw.tape),
            "peak": int(ice_row[C_PEAK]), "walk_split": walk_split,
            "swept": sum(t["row"][3] for t in trades if t["row"][4] == ice_sub)}


def slices(total, peak):
    n = ceil_div(total, peak)
    return [peak] * (n - 1) + [total - (n - 1) * peak]


def closed_form(construction, p):
    # The closed forms of the formulas above, from draw.params only: trades, final books, filled, remaining.
    if construction in (CASCADE, RULE3):
        o_i, o_a = int(p["ice_oid"]), int(p["agg_oid"])
        P, total, peak, q_agg = int(p["p_ice_int"]), int(p["total_int"]), int(p["peak_int"]), int(p["q_agg_int"])
        ppd = 0                                   # every C08 tape: PPD 0 (precondition); slice k at P + k * ppd = P
        b, s = (o_i, o_a) if int(p["ice_side"]) == BUY else (o_a, o_i)
        ice_s = side_name(p["ice_side"])
        if construction == CASCADE:
            xs = slices(total, peak)
            books = [{"BUY": [], "SELL": []}]
        else:
            m, rho = divmod(q_agg, peak)
            xs = [peak] * m + ([rho] if rho else [])
            if rho:
                books = [{"BUY": [], "SELL": [], ice_s: [[P, peak - rho, o_i]]}]
            else:
                books = [{"BUY": [], "SELL": [], ice_s: [[P, min(peak, total - q_agg), o_i]]},
                         {"BUY": [], "SELL": []}]
        trades = [[b, s, P + k * ppd, x, 1] for k, x in enumerate(xs)]
        filled = {str(o_i): sum(xs), str(o_a): sum(xs)}
        remaining = {str(o_i): total - sum(xs), str(o_a): q_agg - sum(xs)}
        return {"trades": trades, "books": books, "filled": filled, "remaining": remaining}
    # incoming: the merge of the crossing levels (counter bids, in priority order) with the incoming iceberg's
    # instance partition (peak, ..., peak, r): one row per (counter bid, instance) pair (item (2); OQ-33 revised)
    o_i, P = int(p["ice_oid"]), int(p["p_ice_int"])
    total, peak = int(p["total_int"]), int(p["peak_int"])
    ppd = int(p.get("ppd_int", 0))                # every C08 tape: PPD 0 (precondition); instance k at P + k * ppd
    inst = slices(total, peak)
    lv = list(zip([int(x) for x in p["level_price_ints"]], [int(x) for x in p["level_qty_ints"]],
                  [int(x) for x in p["resting_oids"]]))
    rest_side = int(p["rest_side"])
    ice_side = other(rest_side)
    lv.sort(key=lambda t: rank_key(rest_side, t[0], 0))
    sub, left_ice, trades, passive, left = len(lv), total, [], [], {}
    k, inst_left = 0, inst[0]
    for (pr, q, o) in lv:
        if left_ice == 0 or not crosses(rest_side, pr, P + k * ppd):
            break
        lq = q
        while lq > 0 and left_ice > 0 and crosses(rest_side, pr, P + k * ppd):
            t = min(lq, inst_left)
            b, s = (o_i, o) if rest_side == SELL else (o, o_i)
            trades.append([b, s, pr, t, sub])
            passive.append(o)
            lq -= t
            inst_left -= t
            left_ice -= t
            if inst_left == 0 and left_ice > 0:
                k += 1
                inst_left = inst[k]
        left[o] = lq
    book = sorted([pr, left.get(o, q), o] for (pr, q, o) in lv if left.get(o, q) > 0)
    if left_ice > 0:                              # item (4): the remainder as instance k_e, showing v_1
        M = total - left_ice
        k_e = M // peak
        u = M - k_e * peak
        v1 = min(peak - u, left_ice)
        rest = [P + k_e * ppd, v1, left_ice - v1]
        own = [[rest[0], v1, o_i]]
    else:
        rest, own = [], []
    books = [{side_name(rest_side): book, side_name(ice_side): own}]
    filled = {str(o): q - left.get(o, q) for (_p, q, o) in lv}
    filled[str(o_i)] = total - left_ice
    remaining = {str(o): left.get(o, q) for (_p, q, o) in lv}
    remaining[str(o_i)] = left_ice
    return {"trades": trades, "books": books, "filled": filled, "remaining": remaining, "passive": passive,
            "rest": rest}


def closed_form_agrees(construction, draw):
    try:
        cf, e = closed_form(construction, draw.params), expectation(construction, draw)
    except Exception:  # noqa: BLE001  (a malformed params dict is a disagreement, recorded by the caller)
        return False
    x = e["exact"]
    ok = (cf["trades"] == x["trades"] and cf["filled"] == x["filled"] and cf["remaining"] == x["remaining"]
          and sorted(map(canon, cf["books"])) == sorted(map(canon, e["allowed"]["final_book"])))
    if construction in INCOMING_LIKE:
        ok = (ok and cf["passive"] == x["passive"] and cf["rest"] == x["incoming_rest"]
              and e["walk_split"] == x["instance_split"])
    return bool(ok)


def precondition(construction, draw):
    # Section 4's dataset precondition, from the tape and params. Returns the list of departures.
    bad, t, p = [], np.asarray(draw.tape), draw.params
    rows = real_rows(t)
    n_rows = 2 if construction in (CASCADE, RULE3) else int(p.get("L", -1)) + 1
    if len(rows) != n_rows:
        return [f"{len(rows)} real rows (expected {n_rows})"]
    if len({int(x[C_CONTRACT]) for x in rows}) != 1:
        bad.append("more than one contract")
    oids, ts = [int(x[C_OID]) for x in rows], [int(x[C_TS]) for x in rows]
    if oids != list(range(1, len(rows) + 1)) or any(a >= b for a, b in zip(ts, ts[1:])):
        bad.append(f"order ids {oids} / tape timestamps {ts} not 1..n / strictly increasing")
    if any(not 0 <= int(x[C_PRICE]) <= PRICE_INT_MAX or not QTY_INT_MIN <= int(x[C_QTY]) <= QTY_INT_MAX for x in rows):
        bad.append("price or quantity outside §2.3")
    if any(int(x[C_VALIDITY]) != GFS or int(x[C_PPD]) != 0 for x in rows):
        bad.append("a row is not GFS, or carries a price increment")
    ice = [x for x in rows if int(x[C_EXEC]) == ICE]
    others = [x for x in rows if int(x[C_EXEC]) != ICE]
    if len(ice) != 1 or any(int(x[C_EXEC]) != NON or int(x[C_PEAK]) != 0 for x in others):
        return bad + ["expected one ICE row and NON rows without a peak"]
    ice = ice[0]
    total, peak = int(ice[C_QTY]), int(ice[C_PEAK])
    n = ceil_div(total, peak) if peak > 0 else 0
    n_min = 1 if construction == REMAINDER else 2      # a remainder draw may hold a single instance (total = peak)
    if not 0 < peak <= total or (peak == total and construction != REMAINDER) or not n_min <= n <= SLICE_MARGIN:
        bad.append(f"iceberg peak {peak}, total {total}, ceil(total/peak) {n}")
    if [int(ice[C_OID]), int(ice[C_SIDE]), int(ice[C_PRICE]), total, peak] != [
            int(p.get(k, -1)) for k in ("ice_oid", "ice_side", "p_ice_int", "total_int", "peak_int")]:
        bad.append("params ice_oid / ice_side / p_ice_int / total_int / peak_int != tape")
    if int(p.get("n_slices", -1)) != n:
        bad.append(f"params n_slices {p.get('n_slices')} != ceil(total/peak) {n}")
    if construction in (CASCADE, RULE3):
        a = others[0]
        if [int(ice[C_OID]), int(a[C_OID])] != [1, 2]:
            bad.append("iceberg is not submission 0 or the aggressor is not submission 1")
        if int(a[C_SIDE]) != other(ice[C_SIDE]) or int(p.get("agg_side", -1)) != int(a[C_SIDE]):
            bad.append("aggressor side")
        if [int(a[C_OID]), int(a[C_PRICE]), int(a[C_QTY])] != [
                int(p.get(k, -1)) for k in ("agg_oid", "p_agg_int", "q_agg_int")]:
            bad.append("params agg_oid / p_agg_int / q_agg_int != tape")
        if not crosses(ice[C_SIDE], ice[C_PRICE], a[C_PRICE]) or int(a[C_PRICE]) == int(ice[C_PRICE]):
            bad.append("the aggressor's limit does not cross strictly through p_ice")
        if int(p.get("remainder_int", -1)) != total - (n - 1) * peak:
            bad.append("params remainder_int != total - (n - 1) * peak")
        if construction == CASCADE and int(a[C_QTY]) != total:
            bad.append("cascade: q_agg != total")
        if construction == RULE3:
            if n < 3 or not peak < int(a[C_QTY]) <= total - peak:
                bad.append(f"rule3: n {n}, q_agg {int(a[C_QTY])} not in (peak, total - peak]")
            if abs(int(a[C_PRICE]) - int(ice[C_PRICE])) != int(p.get("through_ticks", -1)):
                bad.append("rule3: |L - p_ice| != params through_ticks")
    else:
        lv = others
        rs = int(p.get("rest_side", -1))
        if int(ice[C_OID]) != len(rows) or any(int(x[C_SIDE]) != rs for x in lv) or int(ice[C_SIDE]) != other(rs):
            bad.append("incoming: the iceberg is not the last submission, or sides")
        prices = [int(x[C_PRICE]) for x in lv]
        if [prices, [int(x[C_QTY]) for x in lv], [int(x[C_OID]) for x in lv]] != [
                [int(v) for v in p.get(k, [])] for k in ("level_price_ints", "level_qty_ints", "resting_oids")]:
            bad.append("params level_price_ints / level_qty_ints / resting_oids != tape")
        worse = all((b > a) if rs == SELL else (b < a) for a, b in zip(prices, prices[1:]))
        if len(set(prices)) != len(prices) or not worse:
            bad.append("level prices not pairwise distinct and strictly worsening")
        if int(ice[C_PRICE]) not in prices:
            bad.append("p_ice is not a level price")
        if construction == INCOMING:
            if sum(int(x[C_QTY]) for x in lv if crosses(rs, x[C_PRICE], ice[C_PRICE])) < total:
                bad.append("the crossing levels hold less than total")
        else:   # REMAINDER: every level crosses, depth M < total, M mod peak != 0, no price increment
            depth = sum(int(x[C_QTY]) for x in lv)
            if not all(crosses(rs, x[C_PRICE], ice[C_PRICE]) for x in lv):
                bad.append("remainder: a level does not cross p_ice")
            if not depth < total or depth % peak == 0:
                bad.append(f"remainder: depth {depth} not < total {total}, or a multiple of peak {peak}")
            if int(p.get("ppd_int", -1)) != 0:
                bad.append("remainder: params ppd_int != 0")
    if max_slices(t) > SLICE_MARGIN:
        bad.append(f"an iceberg row with ceil(qty/peak) > {SLICE_MARGIN}")
    return bad


# ---- the expectation on the first draw of each construction ---------------------------------------------------
CF_AGREES = True
for _c, _n, _s0 in PLAN:
    if FIRST.get(_c) is None:
        continue
    _d = DRAWN[(_c, FIRST[_c])]
    _e = expectation(_c, _d)
    _ok = closed_form_agrees(_c, _d)
    CF_AGREES &= _ok
    print(f"{_c}, seed {FIRST[_c]} [{_e['class']}]: n = ceil(total/peak) = {_e['n_slices']}")
    print(f"    (Q) trades [buyer, seller, price_int, qty_int, submission]: {_e['exact']['trades']}")
    print(f"    (S) trade sizes: {[t[3] for t in _e['exact']['trades']]}")
    print(f"    (F) filled {_e['exact']['filled']}; remaining {_e['exact']['remaining']}")
    print(f"    (B) final book one of: {_e['allowed']['final_book']}")
    print(f"    (E) reports {_e['exact']['reports']}; rows per submission {_e['exact']['rows_qty']}")
    if "new_slice_ts" in _e["allowed"]:
        print(f"    (N) resting slice timestamp vs first slice: one of {_e['allowed']['new_slice_ts']}")
    if "instance_split" in _e["exact"]:
        print(f"    (I) instance split (walk): {_e['walk_split']}; rows {len(_e['exact']['trades'])} for "
              f"{len(set(_e['exact']['passive']))} levels touched")
        print(f"    (R) rested remainder [price, visible, hidden]: {_e['exact']['incoming_rest']}")
    print(f"    closed form == walk: {_ok}; precondition departures: {precondition(_c, _d)}")
print(f"closed form agrees with (W) on the first draw of each construction: {CF_AGREES}")

## 6. Run

### Eager and jitted (wired at K3)

> **Wired at K3.** `run_draw` below is the battery's record builder on the draw, exactly as the contract states. Sections 1–5 and 7 were not edited (independence rule 2). The three C08 constructions in `omie_tape_synth.CONSTRUCTIONS` are run as drawn; the fourth, `C08.incoming_remainder`, comes from the generator variant added at the r4 K3 (`make_iceberg_cascade(seed, variant="incoming_remainder")`, prepared in section 3) and is run the same way. `run_draw` needed no change for it.

Section 7 consumes exactly **one name** from this section. If it were not callable, section 7 would record a failing `section 6 wired` check and the notebook would end `NOT CERTIFIED`.

**`run_draw(draw, *, jit) -> RunRecord`**: the battery's record builder on one draw, i.e. `tests.omiecid.reference.omie_invariants.record_from_draw(draw, jit=jit)`, with no other argument. It runs `draw.tape` **as is** (nothing in it may be rewritten) on one fresh book of an OPEN contract with the default configs (`OMIEProductConfig()`, `OMIEMarketConfig()`), the draw's own analytic capacity bounds (`draw.capacity_bounds`), no cancellations and no gate-closure expiry. `jit=False` runs eager (the builder wraps the run in `jax.disable_jit()`); `jit=True` runs jitted.

Section 7 reads these `RunRecord` fields:
- `fills`: each `Fill`'s `submission`, `row_index`, `buyer_oid`, `seller_oid`, `price_int`, `qty_int`, `passive_oid`, `aggressor_oid`, `sec`, `nsec`;
- `accepted` (its keys) and `rejections` (each `Rejection.oid`);
- `filled`, `still_resting`, `hidden_remaining`, `evicted` and `cancelled_or_expired` (each `Cancellation.qty_int`), per order id;
- `post_snapshots[s].resting(side)` and `final_snapshot.resting(side)`: `(price, qty, oid, agent, sec, nsec)`;
- `engine_reports`: each `EngineReport`'s `submission`, `accepted`, `traded`, `trade_qty_mw`, `trade_price_eur` and `settlement_value_eur`;
- `observed_capacity`;
- the whole record, through `run_battery` (and its I3, I6 and I8 results) and `compare_records`.

Section 7 calls `run_draw(..., jit=True)` on every draw it uses, and `run_draw(..., jit=False)` on the eager selection of section 3. A seed beyond the initial ranges (a re-draw after a discard) is prepared with `prepare(construction, seed)` before the call.

In [ ]:
# ---------------------------------------------------------------------------------------
# Section 6 (K3): run_draw(draw, *, jit) -> RunRecord, per the contract above.
#
# The battery's record builder on one draw, with no other argument:
# record_from_draw runs draw.tape as is on one fresh book of one OPEN contract, with the
# default configs (OMIEProductConfig(), OMIEMarketConfig()), the draw's own analytic
# capacity bounds (draw.capacity_bounds), no cancellations (no C08 draw schedules one) and
# no gate-closure expiry. jit=False wraps the whole run in jax.disable_jit() (inside
# build_record); jit=True runs it jitted. Every call starts from its own fresh book.
# ---------------------------------------------------------------------------------------
from tests.omiecid.reference.omie_invariants import record_from_draw


def run_draw(draw, *, jit):
    """One run of `draw.tape` (unmodified) on a fresh OPEN book: the battery's record builder."""
    return record_from_draw(draw, jit=jit)


print("run_draw: record_from_draw(draw, jit=jit) on one OPEN contract, default configs, the draw's capacity "
      "bounds, no cancels, no GCT expiry; jitted on every draw, eager on the section-3 selection")

## 7. Certification

The oracle comparison comes first, by order id and in integer ticks, one combined assertion per valid draw; then the invariant battery (RB, I1–I9) as a pass/fail table, the capacity peaks and the discard rate. No check stops the notebook: every assertion is evaluated and recorded.

**What is recorded, per valid draw.** One assertion named by seed, construction and class, whose `expected` and `observed` are the canonical compact JSON (sorted keys) of section 5's values **in full**, never truncated, so an auditor can check every integer from the results JSON alone: accepted and rejected ids; every trade row `[buyer, seller, price_int, qty_int, submission]` in row order (and the passive id per row on the incoming draw); filled and remaining per id; evicted / cancelled quantities (`lost`, expected empty); the final book; each submission's report and the quantity of the rows it wrote; and, where they apply, the reported-price bracket, the new-slice timestamp, and on the incoming draw the instance-split check (I) and the rested remainder (R). Where section 5 lists a set (the final book under F4, the new-slice timestamp), `expected` holds the whole set and the observed value must be one of its members. The assertion's `detail` lists any key that differs, then, **reported and not asserted**: the hop-shaped rows (passive = the submitting order's own id), the distinct trade-row timestamps per traded submission, the reported price in ticks and in euros, and the reported settlement beside the value of the rows ($k\,D_s$) and the battery's tolerance $\tau_s$. The draw's tape and parameters are kept for a frozen counterexample (`record_case`).

**Aggregate assertions:** the section-6 wiring; the closed form against (W), on the first draws and on every valid draw; the dataset precondition (section 4), listing every departing seed and its departures; **I3** and **I6** hold on every valid draw of each construction — `not_applicable` counts as vacuously satisfied, only `fail` fails (finding F12; failing seeds listed in full); **no capacity flag inside the margin** (finding F7), listing every I8-discarded seed that had passed the pre-run margin; the **hop bound** ≤ 100, the real ceiling of 100 outer matching passes (finding F11; the generator envelope's 160 is not the ceiling), and the largest $\lceil qty/peak \rceil$ ≤ 64 on every valid draw, with the largest values reported; **coverage** — all fourteen classes among the valid draws; **instance-split non-vacuity** — at least one valid incoming draw splits a level's fill at an incoming instance boundary (finding F16), seeds listed in full; **remainder non-vacuity** — at least one valid incoming-remainder draw with $R > 0$ and $u > 0$ (finding F14), seeds listed in full, the variant's generator digest in its detail; the eager comparisons, at least one per class and per construction, a missing one recorded as a failure.

The battery runs on every jitted record with `expects_trade = True` (every C08 draw trades). A draw that I8 marks `invalid` is discarded and replaced by the next seed. **Reported, not asserted:** the valid draws per class, the discards per construction and reason, the number of hop-shaped rows observed, and, per incoming construction, the numbers of draws with $R > 0$, $u > 0$ and $v_1 < R$ (expected 0 on `C08.incoming_iceberg`; finding F14).

In [ ]:
import math
import traceback
from collections import defaultdict

from tests.omiecid.reference.omie_invariants import compare_records, i6_tolerance, run_battery

cert = Certification(
    cert_id=CERT_ID, notebook=NOTEBOOK, scenario="Iceberg regeneration",
    known_result="cascade: one aggressor of size total consumes all n = ceil(total/peak) slices of a resting iceberg in "
                 "its own submission, trades [buyer, seller, p_ice, x_k, 1] with sizes peak,...,peak,r; Rule-3: an "
                 "aggressor peak < q_agg <= total - peak, limit L strictly through p_ice, trades peak x floor(q_agg/peak) "
                 "then q_agg mod peak, every one at p_ice (OQ-31 Reading B), never at L; the iceberg keeps total - q_agg "
                 "and its resting slice carries a later timestamp; incoming: an iceberg walking a resting ladder aggresses with "
                 "its total as a chain of size-peak instances, one trade per (counter bid, instance) pair: rows split at "
                 "resting entries and at its own instance boundaries (every row <= peak, every multiple of peak below M "
                 "a cumulative row boundary), each at the level's limit, filled total; incoming remainder: the same "
                 "iceberg against levels of total depth M < total, M mod peak != 0, sweeps every level and rests "
                 "R = total - M as instance k_e = floor(M/peak) at p_ice showing v_1 = min(peak - u, R), "
                 "u = M - k_e*peak, hidden R - v_1 (v2.14 Rule 2 Incoming iceberg items (2), (4), amended "
                 "2026-10-20; OQ-33 revised, OQ-34 resolved, display C3); "
                 "every report equals the rows its submission wrote (engineering invariant, I6(b)); book uncrossed at "
                 "every return (I3); hop bound <= 100 outer matching passes; no capacity flag (truncated = False) inside the "
                 "margin; no iceberg_meta claim",
    oracle_kind="exact-parametric", quotes=QUOTES, citations=CITATIONS,
    # emit() digests only CONSTRUCTIONS keys; C08.incoming_remainder's digest is recorded in its aggregate check
    constructions=tuple(c for c, _, _ in PLAN if c in synth.CONSTRUCTIONS),
    requested_draws=TOTAL_DRAWS, runtime_forecast_s=480.0, started_at=T0,
    downstream="C09 (ICEwPI slice pricing builds on the same regeneration and price-setter rule), C10 (FOK multi-slice "
               "fills and hidden-volume accounting, C10b/C10c), C11 (GCT expiry of resting icebergs' hidden remainder), "
               "C12 (settlement of multi-slice submissions: the I6(b) report), C15 (fuzz sessions with icebergs); "
               "invariants I3, I5 (Rule-3 hop branch) and I6(b); Track B, whose replayed books contain icebergs",
)

R_SEQ = ("§4.2.1 Rules 2-4 (v2.14 Incoming iceberg, Re-ranking); §4.2; §6.2 items 2, 3, 6, 7; §5.2 iceberg-slice "
         "bullet; OQ-31 (Rule-3 price = p_ice); OQ-33 revised 2026-10-20 (rows split at resting entries and "
         "incoming instance boundaries); OQ-34 resolved 2026-10-20 (remainder as instance k_e showing "
         "v_1 = min(P - u, R), hidden R - v_1; display C3 derived from data)")
R_REP = "ENGINEERING INVARIANT (report equals tape, I6(b); report price bracket)"
R_I3 = "§4.2.1 Rule 4 (a crossed order book cannot persist); I3"
R_CAP = ("capacity-validity precondition (five ceilings; 100 outer matching passes, ladder v2.4 / S7); card Known "
         "result truncated = False")
R_DATA = "dataset precondition (section 4); §2.3"
R_EAGER = "eager == jitted (scenario-ladder: what eager and jitted mean here)"
OFFSET = int(round(-CONFIG.price_min / CONFIG.price_tick))


def guarded(label, seed, fn, *args, **kw):
    # Call a section-6 function; a raise is recorded as a failed check, never propagated.
    try:
        return fn(*args, **kw)
    except Exception as exc:  # noqa: BLE001
        cert.check(f"{label}: run raised", False, seed=seed, expected="a RunRecord", observed=type(exc).__name__,
                   detail=f"{type(exc).__name__}: {exc} | " + traceback.format_exc().splitlines()[-1], rule=R_SEQ)
        return None


def to_ticks(x, origin, tick):
    # A float report figure in ticks (round to nearest); None when it is not a finite number.
    try:
        v = float(x)
    except (TypeError, ValueError):
        return None
    return int(round((v - float(origin)) / float(tick))) if math.isfinite(v) else None


def book3(snap):
    return {side_name(sd): sorted([int(x[0]), int(x[1]), int(x[2])] for x in snap.resting(sd)) for sd in (BUY, SELL)}


def ts_of(snap, oid, side):
    rows = [(int(x[4]), int(x[5])) for x in snap.resting(side) if int(x[2]) == int(oid)]
    return rows[0] if len(rows) == 1 else None


def observe(rec, draw, e):
    # The observed counterpart of e["exact"] and e["allowed"], plus the reported-only info.
    fills = sorted(rec.fills, key=lambda f: (int(f.submission), int(f.row_index)))
    oids, n_sub = e["oids"], len(e["oids"])
    o = {
        "accepted": sorted(int(x) for x in rec.accepted),
        "rejected": sorted(int(r.oid) for r in rec.rejections),
        "trades": [[int(f.buyer_oid), int(f.seller_oid), int(f.price_int), int(f.qty_int), int(f.submission)]
                   for f in fills],
        "filled": {str(x): int(rec.filled.get(x, 0)) for x in oids},
        "remaining": {str(x): int(rec.still_resting.get(x, 0)) + int(rec.hidden_remaining.get(x, 0)) for x in oids},
        "lost": {str(x): [int(rec.evicted.get(x, 0)), sum(int(c.qty_int) for c in rec.cancelled_or_expired.get(x, ()))]
                 for x in sorted(set(rec.evicted) | set(rec.cancelled_or_expired))
                 if int(rec.evicted.get(x, 0)) or sum(int(c.qty_int) for c in rec.cancelled_or_expired.get(x, ()))},
        "final_book": book3(rec.final_snapshot),
    }
    ers = sorted(rec.engine_reports, key=lambda r: int(r.submission))
    o["reports"] = [[bool(r.accepted), bool(r.traded), to_ticks(r.trade_qty_mw, 0.0, CONFIG.qty_tick)] for r in ers]
    o["rows_qty"] = [sum(int(f.qty_int) for f in fills if int(f.submission) == s) for s in range(n_sub)]
    if "passive" in e["exact"]:
        o["passive"] = [int(f.passive_oid) for f in fills]
    if "instance_split" in e["exact"]:
        o["instance_split"] = instance_split([int(f.qty_int) for f in fills if int(f.submission) == e["ice_sub"]],
                                             e["peak"])
    if "incoming_rest" in e["exact"]:
        _rows = [x for x in rec.final_snapshot.resting(e["ice_side"]) if int(x[2]) == e["ice_oid"]]
        _hid = int(rec.hidden_remaining.get(e["ice_oid"], 0))
        o["incoming_rest"] = ([int(_rows[0][0]), int(_rows[0][1]), _hid] if len(_rows) == 1
                              else [] if not _rows and _hid == 0
                              else {"rows": [[int(x[0]), int(x[1])] for x in _rows], "hidden": _hid})
    price_ticks = [to_ticks(r.trade_price_eur, CONFIG.price_min, CONFIG.price_tick) for r in ers]
    if e["agg_sub"] is not None:
        pt = price_ticks[e["agg_sub"]] if len(price_ticks) > e["agg_sub"] else None
        o["report_price_in_bracket"] = pt is not None and e["p_ice"] - 1 <= pt <= e["p_ice"] + 1
    if "new_slice_ts" in e["allowed"]:
        first = (ts_of(rec.post_snapshots[e["ice_sub"]], e["ice_oid"], e["ice_side"])
                 if len(rec.post_snapshots) > e["ice_sub"] else None)
        last = ts_of(rec.final_snapshot, e["ice_oid"], e["ice_side"])
        o["new_slice_ts"] = ("absent" if last is None else "no first-slice row" if first is None
                             else "later" if last > first else "not later")
        o_ts = {"first_slice": first, "resting_slice": last}
    else:
        o_ts = None
    sub_oid = {s: oids[s] for s in range(n_sub)}
    info = {
        "hop_shaped_rows": sum(1 for f in fills if int(f.passive_oid) == sub_oid.get(int(f.submission))),
        "trade_ts_distinct": {str(s): len({(int(f.sec), int(f.nsec)) for f in fills if int(f.submission) == s})
                              for s in range(n_sub) if any(int(f.submission) == s for f in fills)},
        "report_price_int": price_ticks,
        "report_price_eur": [round(float(r.trade_price_eur), 6) for r in ers],
        "report_settlement_eur": [round(float(r.settlement_value_eur), 6) for r in ers],
        "rows_value_eur_and_tau": {str(s): [round(v, 6), float(f"{tau:.3e}")] for s, (tau, v) in (
            (s, i6_tolerance([(int(f.price_int), int(f.qty_int)) for f in fills if int(f.submission) == s], CONFIG,
                             chunks=len({(int(f.sec), int(f.nsec)) for f in fills if int(f.submission) == s})))
            for s in range(n_sub))},
    }
    if o_ts is not None:
        info["slice_ts"] = o_ts
    return o, info


def check_draw(construction, seed, rec, draw, e):
    # (A), (Q)+(S), (F), (B), (N), (E) of section 5, on the jitted record. Returns True when all passed.
    o, info = observe(rec, draw, e)
    diff = [k for k, v in e["exact"].items() if o.get(k) != v]
    diff += [k for k, allowed in e["allowed"].items() if o.get(k) not in allowed]
    expected = {**e["exact"], **{f"{k}_one_of": v for k, v in e["allowed"].items()}}
    return cert.check(
        f"seed {seed} [{e['class']}]: exact trade sequence, slice sizes, fills, final book, reports",
        not diff, seed=seed, expected=canon(expected), observed=canon(o), unit="other",
        detail=("differs on: " + ", ".join(diff) + " | " if diff else "") + "reported only: " + canon(info),
        rule=R_SEQ + "; " + R_REP)


EAGER_DONE = defaultdict(list)   # class or construction -> seeds with an eager comparison


def eager_compare(construction, seed, draw, rec, tags):
    # Run the draw eager and compare it with its jitted record (one eager_jit assertion).
    rec_e = guarded(f"seed {seed} {SHORT[construction]} (eager)", seed, run_draw, draw, jit=False)
    diff = (["jitted run missing" if rec is None else "eager run missing"] if rec is None or rec_e is None
            else list(compare_records(rec_e, rec)))
    for t in tags:
        EAGER_DONE[t].append(seed)
    cert.check(f"eager == jitted (seed {seed}, {SHORT[construction]})", diff == [], kind="eager_jit", seed=seed,
               expected=True, observed=diff == [], unit="bool", detail=", ".join(diff) if diff else "identical",
               rule=R_EAGER)


def margin_breaches(tape, bounds):
    # The capacity-validity precondition (five ceilings), checked from the tape before it is run.
    lim = {"resting_per_side": ENVELOPE.max_resting_per_side, "trades": ENVELOPE.max_trades,
           "hops": ENVELOPE.max_hops, "concurrent_icebergs": ENVELOPE.max_concurrent_icebergs}
    bad = []
    for name, bd in (("draw", bounds), ("generic", synth.tape_capacity_bounds(tape))):
        bad += [f"{name} {k} bound {bd.get(k)} > {m}" for k, m in lim.items() if int(bd.get(k, 0)) > m]
        if int(bd.get("hops", 0)) > PASS_BOUND:      # the real ceiling: 100 outer matching passes (finding F11)
            bad.append(f"{name} hop bound {bd.get('hops')} > {PASS_BOUND} passes")
    if max_slices(tape) > SLICE_MARGIN:
        bad.append(f"ceil(qty/peak) {max_slices(tape)} > {SLICE_MARGIN}")
    return bad


cert.check("section 5: closed form agrees with the walk (W) on the first draw of each construction", CF_AGREES,
           expected=True, observed=CF_AGREES, unit="bool", rule=R_SEQ)
wired = callable(globals().get("run_draw"))
cert.check("section 6 wired (run_draw defined)", wired, expected=True, observed=wired, unit="bool",
           detail="" if wired else "section 6 is still the K2 stub", rule=R_SEQ)

PER_RUN, DATA_BAD, CLOSED_BAD = [], {}, defaultdict(list)
VALID = defaultdict(list)                  # construction -> valid seeds
COVER = Counter()
FLAG_INSIDE_MARGIN = {}                    # seed -> I8 evidence, for draws that passed the pre-run margin
BAT_FAIL = defaultdict(lambda: defaultdict(list))   # construction -> check id -> failing seeds (I3, I6)
DISCARDS = defaultdict(Counter)
PEAK = {"hop_bound": 0, "generic_hop_bound": 0, "max_slices": 0}
HOPS_SEEN = Counter()
SPLIT_AT_INSTANCE = []   # valid incoming seeds where a level's fill is split at an incoming instance boundary (F16)
REM_STATS = defaultdict(lambda: {"R>0": [], "u>0": [], "v1<R": []})   # per incoming construction (F14)
WITH_REMAINDER = []      # valid incoming-remainder seeds with R > 0 and u > 0 (F14; asserted non-empty)


def run_construction(construction, quota, seed0):
    valid, seed = 0, seed0
    while valid < quota and seed < seed0 + 2 * quota:
        draw = DRAWN.get((construction, seed), "missing")
        if draw == "missing":
            draw = prepare(construction, seed)
        if draw is None:
            cert.discard(seed, f"{construction}: generator refused the seed: {REFUSED.get((construction, seed))}")
            DISCARDS[construction]["generator refused"] += 1
            seed += 1
            continue
        margin = margin_breaches(draw.tape, dict(draw.capacity_bounds))
        if margin:
            cert.discard(seed, f"{construction}: pre-run capacity margin: " + "; ".join(margin))
            DISCARDS[construction]["pre-run margin"] += 1
            seed += 1
            continue
        bad = precondition(construction, draw)
        if bad:
            DATA_BAD[f"{construction} seed {seed}"] = bad
            cert.check(f"seed {seed} ({SHORT[construction]}): draw is as section 4 describes", False, seed=seed,
                       expected=[], observed=bad, unit="other", rule=R_DATA)
            seed += 1
            continue
        e = expectation(construction, draw)
        rec = guarded(f"seed {seed} {SHORT[construction]} (jit)", seed, run_draw, draw, jit=True)
        if rec is None:
            seed += 1
            continue
        rep = run_battery(rec, expects_trade=True)
        if not cert.add_battery(rep, seed=seed):
            i8 = next(r for r in rep.results if r.check_id == "I8")
            FLAG_INSIDE_MARGIN[f"{construction} seed {seed}"] = dict(i8.evidence)
            DISCARDS[construction]["I8 invalid"] += 1
            seed += 1   # discarded and re-drawn with the next seed (and a failed assertion: finding F7)
            continue
        valid += 1
        VALID[construction].append(seed)
        cert.record_case(seed, draw.tape, {**draw.params, "construction": construction})
        PEAK["hop_bound"] = max(PEAK["hop_bound"], int(draw.hop_bound))
        PEAK["generic_hop_bound"] = max(PEAK["generic_hop_bound"], int(synth.tape_capacity_bounds(draw.tape)["hops"]))
        PEAK["max_slices"] = max(PEAK["max_slices"], max_slices(draw.tape))
        COVER[e["class"]] += 1
        if not closed_form_agrees(construction, draw):
            CLOSED_BAD[construction].append(seed)
        for r in rep.results:
            if r.check_id in ("I3", "I6") and r.status == "fail":   # not_applicable is vacuously satisfied (F12)
                BAT_FAIL[construction][r.check_id].append(seed)
        cert.record_capacity({**rec.observed_capacity, "hops": None})   # hops: unobserved by the engine
        ok = check_draw(construction, seed, rec, draw, e)
        PER_RUN.append({"construction": construction, "seed": seed, "class": e["class"], "ok": ok})
        HOPS_SEEN[construction] += sum(1 for f in rec.fills if int(f.passive_oid) == e["oids"][int(f.submission)])
        if construction == INCOMING and len(e["exact"]["passive"]) > len(set(e["exact"]["passive"])):
            SPLIT_AT_INSTANCE.append(seed)
        if construction in INCOMING_LIKE and e["exact"]["incoming_rest"]:
            _R = e["exact"]["incoming_rest"][1] + e["exact"]["incoming_rest"][2]
            _u = e["swept"] % e["peak"]
            REM_STATS[construction]["R>0"].append(seed)
            if _u > 0:
                REM_STATS[construction]["u>0"].append(seed)
            if e["exact"]["incoming_rest"][1] < _R:
                REM_STATS[construction]["v1<R"].append(seed)
            if construction == REMAINDER and _u > 0:
                WITH_REMAINDER.append(seed)
        if valid <= EAGER_DRAWS or e["class"] not in EAGER_DONE:
            eager_compare(construction, seed, draw, rec, [e["class"], construction])
        seed += 1
    return valid


N_VALID = {c: (run_construction(c, n, s0) if wired else 0) for c, n, s0 in PLAN}

# ---- aggregate assertions (every list in full) ----------------------------------------------------------------
_cb = {c: CLOSED_BAD.get(c, []) for c, _, _ in PLAN}
cert.check("section 5: closed form agrees with the walk (W) on every valid draw", wired and not any(_cb.values()),
           expected={c: [] for c in _cb}, observed=_cb, unit="other", rule=R_SEQ)
cert.check("dataset: every draw is as section 4 describes (precondition)", wired and not DATA_BAD,
           expected={}, observed=DATA_BAD, unit="other",
           detail=f"{len(DATA_BAD)} draws depart" if DATA_BAD else f"{sum(N_VALID.values())} valid draws", rule=R_DATA)
for _cid, _rule in (("I3", R_I3), ("I6", R_REP + "; §8.1")):
    _fails = {c: BAT_FAIL[c][_cid] for c, _, _ in PLAN}
    cert.check(f"{_cid} holds (pass or not_applicable) on every valid draw of each construction", wired and not any(_fails.values()),
               expected={c: [] for c in _fails}, observed=_fails, unit="other",
               detail=f"valid draws: {dict(N_VALID)}", rule=_rule)
cert.check("no capacity flag (truncated, book_full, saturated ring) on a draw inside the analytic margin (F7)",
           wired and not FLAG_INSIDE_MARGIN, expected={}, observed=FLAG_INSIDE_MARGIN, unit="other", rule=R_CAP)
cert.check(f"hop bound <= {PASS_BOUND} outer matching passes on every valid draw (largest reported: draw bound, "
           "generic bound)",
           wired and 0 < PEAK["hop_bound"] <= PASS_BOUND and PEAK["generic_hop_bound"] <= PASS_BOUND,
           expected=f"<= {PASS_BOUND}", observed=[PEAK["hop_bound"], PEAK["generic_hop_bound"]],
           unit="count", rule=R_CAP)
cert.check("largest ceil(qty/peak) over every valid draw's icebergs <= 64 (largest reported)",
           wired and 0 < PEAK["max_slices"] <= SLICE_MARGIN, expected=f"<= {SLICE_MARGIN}",
           observed=PEAK["max_slices"], unit="count", rule=R_CAP)
_got = {c: COVER[c] for c in CLASSES}
_missing = [c for c, v in _got.items() if v == 0]
cert.check("dataset: the valid draws cover every class (section 3)", wired and not _missing,
           expected={c: ">= 1" for c in CLASSES}, observed=_got, unit="count",
           detail=f"missing: {_missing}" if _missing else "", rule=R_DATA)
cert.check("incoming: at least one valid draw splits a level's fill at an incoming instance boundary (F16)",
           wired and len(SPLIT_AT_INSTANCE) > 0, expected=">= 1", observed=len(SPLIT_AT_INSTANCE), unit="count",
           detail=f"seeds: {SPLIT_AT_INSTANCE}", rule=R_DATA + "; §4.2.1 Rule 2 Incoming iceberg item (2); OQ-33 revised")
import hashlib as _hl
_rem_digest = _hl.blake2b("".join(synth.make_iceberg_cascade(int(s), variant="incoming_remainder").digest()
                                  for s in synth.DIGEST_SEEDS).encode("ascii"), digest_size=16).hexdigest()
cert.check("incoming remainder: at least one valid draw rests a remainder R > 0 after a part-used instance u > 0 (F14)",
           wired and len(WITH_REMAINDER) > 0, expected=">= 1", observed=len(WITH_REMAINDER), unit="count",
           detail=f"seeds: {WITH_REMAINDER} | generator digest of {REMAINDER} over DIGEST_SEEDS "
                  f"{list(synth.DIGEST_SEEDS)}: {_rem_digest}",
           rule=R_DATA + "; §4.2.1 Rule 2 Incoming iceberg item (4); OQ-34 resolved (display C3)")

# ---- eager: every class and every construction must have had one comparison (absence = failure) --------------
for _t in list(CLASSES) + [c for c, _, _ in PLAN]:
    if not EAGER_DONE.get(_t):
        cert.check(f"eager == jitted for '{_t}'", False, kind="eager_jit", expected=True, observed=None, unit="bool",
                   detail=f"no eager comparison was run for '{_t}'", rule=R_EAGER)

# ---- reports (not asserted) ------------------------------------------------------------------------------------
_fail_runs = [r for r in PER_RUN if not r["ok"]]
display(Markdown(f"**Per-draw summary.** {len(PER_RUN)} draws checked, {len(_fail_runs)} with a failed assertion"
                 + (": " + ", ".join(f"{SHORT[r['construction']]} seed {r['seed']}" for r in _fail_runs)
                    if _fail_runs else ".")))
print(f"valid draws: {dict(N_VALID)}; per class: {dict(COVER)}")
print(f"discards per construction and reason: { {c: dict(v) for c, v in DISCARDS.items()} }")
print(f"eager comparisons run: { {k: v for k, v in EAGER_DONE.items()} }")
print(f"hop-shaped rows observed (passive = the submitting order's own id; reported only): {dict(HOPS_SEEN)}")
print(f"incoming draws with a level split at an instance boundary: {len(SPLIT_AT_INSTANCE)}; "
      f"remainder counts per incoming construction (F14): "
      f"{ {c: {k: len(v) for k, v in REM_STATS[c].items()} for c in INCOMING_LIKE} }")
print(f"analytic peaks over valid draws: hop bound {PEAK['hop_bound']} (generic {PEAK['generic_hop_bound']}), "
      f"largest ceil(qty/peak) {PEAK['max_slices']}")
display(Markdown(cert.table()))

## 8. Verdict

Emit the results JSON (validated against `results.schema.json`) and print exactly one word:
`PASS` or `NOT CERTIFIED`. On `NOT CERTIFIED` the smallest failing case is frozen as a fixture.

In [ ]:
RESULTS_PATH = cert.emit()
print(cert.verdict_line())
print(f"results -> {RESULTS_PATH}")

In [ ]:
# Section 9 ("Why this certification failed") is rendered here only on NOT CERTIFIED; it is
# persisted as the last markdown cell at K6-F by cert_support.append_failure_section.
_section9 = cert.failure_section_markdown()
if _section9:
    display(Markdown(_section9))